# D3 Aaya — Online / Adaptive GraphRAG Retrieval Comparison (V2 Final Patch)

This notebook continues the existing V2 work. It does **not** restart from scratch.

It compares:

1. `static_graphrag`
2. `topic_gated_graphrag`
3. `feedback_adaptive_graphrag`

Final required outputs:

```text
reports/tables/d3_online_retrieval_comparison.csv
reports/tables/d3_online_retrieval_per_query.csv
reports/tables/d3_online_feedback_events.csv
reports/member_sections/aaya_d3_adaptation_section.md
```

Shared notebook update target:

```text
notebooks/D3_graphrag_eval_safety.ipynb
```

Evaluation policy:

- `strict_chunk_recall@5` is the hardest metric and requires exact gold chunk matching.
- `document_recall@5` and `page_window_recall@5` are fairer GraphRAG evidence metrics.
- `soft_*` metrics are **diagnostic only** and must not be presented as gold relevance.
- Weak results are not hidden. If adaptive does not improve, the notebook reports that directly.


## 1. Configuration

Start with `MAX_EVAL_ROWS = 2`, then use `5`, then `30`.

Use `SAMPLE_STRATEGY = "balanced_topic"` so the first rows are not all from one topic.


In [1]:
# -----------------------------
# Run configuration
# -----------------------------

TOP_K = 5

# Required switch:
# - "smoke": quick test using 2 queries
# - "final": final D3 evidence using 10-20 queries
RUN_MODE = "final"
CLEAR_PREVIOUS_RESULTS = True
DISABLE_LLM_INTENT_CLASSIFIER_FOR_BATCH = True
SLEEP_BETWEEN_GRAPH_RAG_CALLS_SEC = 45
GEMINI_RETRIES_PER_MODEL = 3
GEMINI_BASE_WAIT_SECONDS = 60   # change to "final" for final evidence

if RUN_MODE == "smoke":
    MAX_EVAL_ROWS = 2
elif RUN_MODE == "final":
    MAX_EVAL_ROWS = 15
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'final'.")

# D3 final evaluation must use this file.
# The notebook validates it and stops if it is missing/incomplete.
D3_GOLD_QA_PATH = "data/gold/d3_gold_qa.csv"
D3_GOLD_QA_REQUIRED = True

# Keep topic accuracy honest:
# If true_topic is missing, it is inferred for routing only, but not counted as gold topic accuracy.
INFER_MISSING_TRUE_TOPIC_FOR_ROUTING = True

# Clean run by default. Set False only when resuming after Gemini quota/rate-limit stop.
CLEAR_PREVIOUS_RESULTS = True
RESUME_FROM_CHECKPOINT = not CLEAR_PREVIOUS_RESULTS

# Gemini generation. No extractive/local generation fallback.




# Preferred models based on available quota.
# The Gemini cell dynamically lists API-supported models and keeps only valid ones.
PREFERRED_GEMINI_MODELS = [
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite",
    "gemini-3.5-flash",
    "gemini-3-flash",
]
GEMINI_MODEL_FALLBACKS = []

# Final output status flags. These are overwritten later if validation/run status changes.
evaluation_status = "not_started"
topic_accuracy_basis = "not_evaluated"
peft_qlora_mode = "not_checked"


## 2. Setup and paths

In [2]:
from __future__ import annotations

import inspect
import json
import math
import os
import re
import sys
import time
import types
import traceback
import hashlib
import importlib
from collections import defaultdict
from dataclasses import dataclass, is_dataclass, asdict
from pathlib import Path
from typing import Any

import pandas as pd
import requests

try:
    import nbformat as nbf
except Exception:
    nbf = None

try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {"notebooks", "notebook"}:
    ROOT = ROOT.parent

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

REPORTS_TABLES = ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

MEMBER_SECTIONS_DIR = ROOT / "reports" / "member_sections"
MEMBER_SECTIONS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = REPORTS_TABLES / "d3_online_retrieval_comparison.csv"
PER_QUERY_PATH = REPORTS_TABLES / "d3_online_retrieval_per_query.csv"
FEEDBACK_EVENTS_PATH = REPORTS_TABLES / "d3_online_feedback_events.csv"
MEMBER_SECTION_PATH = MEMBER_SECTIONS_DIR / "aaya_d3_adaptation_section.md"

CONFIG_PATH = ROOT / "configs" / "config.yaml"
D3_GOLD_QA_FULL_PATH = ROOT / D3_GOLD_QA_PATH

try:
    from dotenv import load_dotenv
    env_path = ROOT / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        print("Loaded .env:", env_path)
    else:
        print("No .env file found:", env_path)
except Exception as dotenv_error:
    print("python-dotenv not available or .env not loaded:", repr(dotenv_error))

# Project .env fallback loader: keeps notebooks working when python-dotenv is not installed.
try:
    from src.rag.graphrag_executor import load_local_env as _project_load_local_env
    _project_load_local_env(ROOT)
    if os.environ.get("NEO4J_USERNAME") and not os.environ.get("NEO4J_USER"):
        os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]
    print("Loaded project .env fallback from:", ROOT / ".env")
except Exception as env_fallback_error:
    print("Project .env fallback not loaded:", repr(env_fallback_error))

# Safety helper: never print API keys from exception URLs in notebook outputs.
def redact_sensitive(value: object) -> str:
    text = str(value)
    for secret_name in ["GEMINI_API_KEY", "GOOGLE_API_KEY", "NEO4J_PASSWORD"]:
        secret = os.environ.get(secret_name)
        if secret:
            text = text.replace(secret, f"[{secret_name}_REDACTED]")
    text = re.sub(r"https://generativelanguage.googleapis.com/[^\s)]+", "https://generativelanguage.googleapis.com/[REDACTED_URL]", text)
    text = re.sub(r"([?&]key=)[^\s)]+", r"\1[REDACTED_API_KEY]", text)
    text = re.sub(r"key=[A-Za-z0-9._\-]+", "key=[REDACTED_API_KEY]", text)
    return text

print("Current working directory:", Path.cwd())
print("Project root:", ROOT)
print("Config exists:", CONFIG_PATH.exists())
print("Sample chunks exists:", (ROOT / "data" / "sample" / "sample_chunks.json").exists())
print("D3 gold Q/A path:", D3_GOLD_QA_FULL_PATH)
print("D3 gold Q/A exists:", D3_GOLD_QA_FULL_PATH.exists())
print("Reports/tables:", REPORTS_TABLES)
print("Member section path:", MEMBER_SECTION_PATH)

print("\nEnvironment check:")
for key in ["NEO4J_URI", "NEO4J_USER", "NEO4J_USERNAME", "NEO4J_DATABASE", "NEO4J_PASSWORD", "GEMINI_API_KEY", "GOOGLE_API_KEY"]:
    value = os.environ.get(key)
    if key.endswith("PASSWORD") or "KEY" in key:
        print(f"- {key}:", "SET" if value else "MISSING")
    else:
        print(f"- {key}:", value if value else "MISSING")

if CLEAR_PREVIOUS_RESULTS:
    for p in [PER_QUERY_PATH, FEEDBACK_EVENTS_PATH, OUTPUT_PATH]:
        if p.exists():
            p.unlink()
            print("Removed old output:", p)


Loaded .env: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\.env
Loaded project .env fallback from: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\.env
Current working directory: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Config exists: True
Sample chunks exists: True
D3 gold Q/A path: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\data\gold\d3_gold_qa.csv
D3 gold Q/A exists: True
Reports/tables: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables
Member section path: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\membe

## 3. D2 continuity

Aaya’s boundary is the online/adaptive routing layer, not re-implementing the retriever. D3 keeps the same boundary but places the adaptive layer inside Rana’s GraphRAG executor.


In [3]:
D2_ONLINE_VS_STATIC = REPORTS_TABLES / "d2_online_vs_static.csv"
D2_SUMMARY = REPORTS_TABLES / "d2_online_adaptation_summary.csv"

if D2_ONLINE_VS_STATIC.exists():
    d2_online_vs_static = pd.read_csv(D2_ONLINE_VS_STATIC)
    print("Loaded:", D2_ONLINE_VS_STATIC)
    display(d2_online_vs_static.head())
else:
    print("Not found:", D2_ONLINE_VS_STATIC)

if D2_SUMMARY.exists():
    d2_summary = pd.read_csv(D2_SUMMARY)
    print("Loaded:", D2_SUMMARY)
    display(d2_summary.head())
else:
    print("Not found:", D2_SUMMARY)


Loaded: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_vs_static.csv


,method,Topic source,BM25 weight source,Recall@5,MRR@5,nDCG@5,Answer citation hit rate,Prequential topic accuracy,Mean latency ms,Final mean BM25 weight,Final mean dense weight,Notes
0,adaptive,River prediction,FeedbackAdapter per-topic weights,0.5800,0.4482,0.4821,0.3533,0.7833,220.1345,0.4801,0.5199,Aaya River + FeedbackAdapter online adaptation
1,static,none,fixed 0.50 / 0.50,0.5767,0.4502,0.4809,0.3517,0.7833,244.4913,0.5000,0.5000,Salma hybrid retriever baseline


Loaded: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d2_online_adaptation_summary.csv


,method,Topic source,BM25 weight source,Recall@5,MRR@5,nDCG@5,Answer citation hit rate,Prequential topic accuracy,Mean latency ms,Final mean BM25 weight,Final mean dense weight,Notes
0,adaptive,River prediction,FeedbackAdapter per-topic weights,0.5800,0.4482,0.4821,0.3533,0.7833,220.1345,0.4801,0.5199,Aaya River + FeedbackAdapter online adaptation
1,static,none,fixed 0.50 / 0.50,0.5767,0.4502,0.4809,0.3517,0.7833,244.4913,0.5000,0.5000,Salma hybrid retriever baseline
2,topic_gated,River prediction,fixed topic profile,0.5900,0.4578,0.4899,0.3717,0.7833,271.1312,0.5228,0.4772,Aaya River topic routing on hybrid baseline


## 4. Topic prediction and feedback adapter

The D1 River classifier starts cold, so early predictions may be `None`. To avoid routing as `unknown`, this notebook uses:

```text
River prediction -> if unknown -> keyword fallback
```

The output still records whether the prediction came from River or the keyword fallback.


In [4]:
TOPIC_ALIASES = {
    "policy": "policy_governance",
    "policy_governance": "policy_governance",
    "policy governance": "policy_governance",
    "technology": "technology_innovation",
    "technology_innovation": "technology_innovation",
    "technology innovation": "technology_innovation",
    "climate science": "climate_science",
    "climate_science": "climate_science",
    "uae cop28": "uae_cop28",
    "uae_cop28": "uae_cop28",
    "cop28": "uae_cop28",
    "uae": "uae_cop28",
    "mitigation": "mitigation",
    "adaptation": "adaptation",
}

CANONICAL_TOPICS = [
    "mitigation",
    "adaptation",
    "climate_science",
    "policy_governance",
    "technology_innovation",
    "uae_cop28",
]


def normalize_topic(topic: Any) -> str:
    if topic is None or (isinstance(topic, float) and pd.isna(topic)):
        return "unknown"
    t = str(topic).strip().lower()
    t = t.replace("-", " ").replace("_", " ")
    return TOPIC_ALIASES.get(t, t.replace(" ", "_"))


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    return str(text)


TOPIC_KEYWORDS = {
    "mitigation": [
        "emissions", "carbon", "co2", "net zero", "renewable",
        "decarbon", "mitigation", "fossil", "energy transition", "fire emissions",
        "greenhouse gas", "ghg", "energy efficiency"
    ],
    "adaptation": [
        "adaptation", "resilience", "flood", "drought", "risk",
        "vulnerability", "agriculture", "water scarcity", "coastal",
        "heat stress", "climate impact"
    ],
    "climate_science": [
        "temperature", "warming", "ipcc", "climate model",
        "scenario", "radiative", "precipitation", "climate change",
        "atmospheric", "ocean", "temperature anomaly"
    ],
    "policy_governance": [
        "policy", "governance", "agreement", "regulation",
        "ndc", "law", "government", "cop", "target", "strategy",
        "commitment"
    ],
    "technology_innovation": [
        "technology", "innovation", "hydrogen", "carbon capture",
        "ai", "machine learning", "digital", "aero", "algorithm",
        "optimization", "model architecture", "system"
    ],
    "uae_cop28": [
        "uae", "dubai", "cop28", "global stocktake",
        "sultan al jaber", "united arab emirates"
    ],
}


def infer_topic_from_text(text: str, default: str = "climate_science") -> str:
    text = normalize_text(text).lower()
    scores = {
        topic: sum(1 for kw in keywords if kw in text)
        for topic, keywords in TOPIC_KEYWORDS.items()
    }
    best_topic, best_score = max(scores.items(), key=lambda x: x[1])
    return best_topic if best_score > 0 else default


def infer_true_topic(row: dict) -> str:
    return infer_topic_from_text(
        f"{row.get('query', '')} {row.get('reference_answer', '')}",
        default="climate_science",
    )


def predict_topic_with_fallback(query: str):
    river_topic = None

    try:
        river_topic = topic_model.predict(query)
    except Exception:
        river_topic = None

    river_topic = normalize_topic(river_topic)

    if river_topic == "unknown":
        return infer_topic_from_text(query, default="climate_science"), "keyword_fallback"

    return river_topic, "river"


try:
    from src.learning.river_topic_classifier import RiverTopicClassifier as ProjectRiverTopicClassifier
    print("Loaded project RiverTopicClassifier.")
except Exception as e:
    ProjectRiverTopicClassifier = None
    print("Could not import project RiverTopicClassifier. Using fallback.", repr(e))


class FallbackRiverTopicClassifier:
    def predict(self, query: str):
        return None

    def learn(self, query: str, topic: str) -> None:
        return None


try:
    from src.learning.feedback_adapter import FeedbackAdapter as ProjectFeedbackAdapter
    print("Loaded project FeedbackAdapter.")
except Exception as e:
    ProjectFeedbackAdapter = None
    print("Could not import project FeedbackAdapter. Using fallback.", repr(e))


@dataclass
class LocalFeedbackUpdate:
    topic: str
    helpful: bool
    reason: str
    bm25_weight: float
    dense_weight: float
    n_feedback: int


class LocalFeedbackAdapter:
    def __init__(self, default_bm25_weight=0.50, step=0.05, min_weight=0.10, max_weight=0.90):
        self.default_bm25_weight = default_bm25_weight
        self.step = step
        self.min_weight = min_weight
        self.max_weight = max_weight
        self.topic_weights = defaultdict(lambda: default_bm25_weight)
        self.feedback_counts = defaultdict(int)

    def get_weights(self, query_topic=None):
        topic = query_topic or "global"
        bm25_weight = self.topic_weights[topic]
        return {"bm25_weight": round(bm25_weight, 4), "dense_weight": round(1.0 - bm25_weight, 4)}

    def update(self, helpful: bool, query_topic=None, reason="generic"):
        topic = query_topic or "global"
        current = self.topic_weights[topic]
        self.feedback_counts[topic] += 1

        if helpful:
            new_weight = current
        elif reason in {"missed_exact_policy", "missed_citation", "needs_source_name"}:
            new_weight = current + self.step
        else:
            new_weight = current - self.step

        new_weight = max(self.min_weight, min(self.max_weight, new_weight))
        self.topic_weights[topic] = new_weight

        return LocalFeedbackUpdate(
            topic=topic,
            helpful=helpful,
            reason=reason,
            bm25_weight=round(new_weight, 4),
            dense_weight=round(1.0 - new_weight, 4),
            n_feedback=self.feedback_counts[topic],
        )


def build_topic_classifier():
    if ProjectRiverTopicClassifier is not None:
        try:
            return ProjectRiverTopicClassifier()
        except Exception as e:
            print("Project RiverTopicClassifier failed to initialize. Using fallback.", repr(e))
    return FallbackRiverTopicClassifier()


def build_feedback_adapter():
    if ProjectFeedbackAdapter is not None:
        for kwargs in [{"default_bm25_weight": 0.50}, {}]:
            try:
                adapter = ProjectFeedbackAdapter(**kwargs)
                if hasattr(adapter, "get_weights") and hasattr(adapter, "update"):
                    return adapter
            except Exception:
                pass
    return LocalFeedbackAdapter(default_bm25_weight=0.50)


topic_model = build_topic_classifier()
feedback_adapter = build_feedback_adapter()

print("Topic model:", type(topic_model).__name__)
print("Feedback adapter:", type(feedback_adapter).__name__)


Loaded project RiverTopicClassifier.
Loaded project FeedbackAdapter.
Topic model: RiverTopicClassifier
Feedback adapter: FeedbackAdapter


## 5. Routing profiles

These are routing policies, not metric values. Metrics are computed later from retrieved results.


In [5]:
STATIC_WEIGHTS = {"bm25_weight": 0.50, "dense_weight": 0.50}

TOPIC_ROUTING_PROFILES = {
    "mitigation": {"bm25_weight": 0.55, "dense_weight": 0.45},
    "adaptation": {"bm25_weight": 0.45, "dense_weight": 0.55},
    "climate_science": {"bm25_weight": 0.40, "dense_weight": 0.60},
    "policy_governance": {"bm25_weight": 0.65, "dense_weight": 0.35},
    "technology_innovation": {"bm25_weight": 0.45, "dense_weight": 0.55},
    "uae_cop28": {"bm25_weight": 0.60, "dense_weight": 0.40},
}

GRAPH_PROFILES = {
    "mitigation": "emissions_policy_evidence",
    "adaptation": "risk_resilience_evidence",
    "climate_science": "scientific_evidence",
    "policy_governance": "policy_citation_evidence",
    "technology_innovation": "technology_solution_evidence",
    "uae_cop28": "cop28_uae_evidence",
    "unknown": "balanced_evidence",
}

display(pd.DataFrame([{"topic": t, **w, "graph_profile": GRAPH_PROFILES[t]} for t, w in TOPIC_ROUTING_PROFILES.items()]))


,topic,bm25_weight,dense_weight,graph_profile
0,mitigation,0.55,0.45,emissions_policy_evidence
1,adaptation,0.45,0.55,risk_resilience_evidence
2,climate_science,0.40,0.60,scientific_evidence
3,policy_governance,0.65,0.35,policy_citation_evidence
4,technology_innovation,0.45,0.55,technology_solution_evidence
5,uae_cop28,0.60,0.40,cop28_uae_evidence


## 6. GraphRAG compatibility patches and initialization

In [6]:
USING_GRAPHRAG_EXECUTOR = False
GRAPHRAG_EXECUTOR = None
ge = None

executor_path = ROOT / "src" / "rag" / "graphrag_executor.py"

try:
    executor_text = executor_path.read_text(encoding="utf-8")
    old_line = 'with open(chunk_store, "r") as fh:'
    new_line = 'with open(chunk_store, "r", encoding="utf-8") as fh:'

    if old_line in executor_text and new_line not in executor_text:
        backup_path = executor_path.with_suffix(".py.bak_utf8")
        if not backup_path.exists():
            backup_path.write_text(executor_text, encoding="utf-8")
        executor_path.write_text(executor_text.replace(old_line, new_line), encoding="utf-8")
        print("Patched UTF-8 chunk loading in:", executor_path)
    else:
        print("UTF-8 chunk loading patch not needed.")
except Exception as patch_error:
    print("Could not apply UTF-8 source patch:", repr(patch_error))

try:
    import src.rag.graphrag_executor as ge
    from src.retrieval.dense_retriever import NumpyDenseRetriever

    class DenseRetrieverCompat:
        def __init__(self, embeddings_path=None, chunks=None, **kwargs):
            if chunks is None:
                chunks = []

            if embeddings_path:
                emb_path = Path(embeddings_path)
                if not emb_path.is_absolute():
                    emb_path = ROOT / emb_path

                print("DenseRetrieverCompat embeddings path:", emb_path)
                print("Embeddings file exists:", emb_path.exists())

                if emb_path.exists():
                    try:
                        self.inner = NumpyDenseRetriever.load(chunks, str(emb_path))
                    except TypeError:
                        self.inner = NumpyDenseRetriever.load(str(emb_path), chunks)
                else:
                    self.inner = NumpyDenseRetriever(chunks=chunks)
            else:
                self.inner = NumpyDenseRetriever(chunks=chunks)

        def search(self, query: str, k: int = 5, filters=None):
            return self.inner.search(query=query, k=k, filters=filters)

    ge.DenseRetriever = DenseRetrieverCompat

    try:
        dense_mod = importlib.import_module("src.retrieval.dense_retriever")
        dense_mod.DenseRetriever = DenseRetrieverCompat
    except Exception:
        pass

    try:
        retrieval_mod = importlib.import_module("src.retrieval")
        retrieval_mod.DenseRetriever = DenseRetrieverCompat
    except Exception:
        pass

    print("Calling GraphRAGExecutor.from_config...")
    GRAPHRAG_EXECUTOR = ge.GraphRAGExecutor.from_config(str(CONFIG_PATH))
    USING_GRAPHRAG_EXECUTOR = True

except Exception as e:
    GRAPHRAG_EXECUTOR = None
    USING_GRAPHRAG_EXECUTOR = False
    print("GraphRAGExecutor failed to initialize.")
    print("Error type:", type(e).__name__)
    print("Error message:", redact_sensitive(e))
    print(redact_sensitive(traceback.format_exc()))

print("USING_GRAPHRAG_EXECUTOR:", USING_GRAPHRAG_EXECUTOR)


UTF-8 chunk loading patch not needed.
Calling GraphRAGExecutor.from_config...
USING_GRAPHRAG_EXECUTOR: True


## 7. Neo4j graph health

In [7]:
GRAPH_DB_READY = False
GRAPH_NODE_COUNT = 0

try:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
    )
    neo4j_database = os.environ.get("NEO4J_DATABASE")
    session_kwargs = {"database": neo4j_database} if neo4j_database else {}

    with driver.session(**session_kwargs) as session:
        ok = session.run("RETURN 1 AS ok").single()["ok"]
        GRAPH_NODE_COUNT = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        labels = session.run("CALL db.labels() YIELD label RETURN collect(label) AS labels").single()["labels"]
        rels = session.run("CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS rels").single()["rels"]

    driver.close()

    GRAPH_DB_READY = GRAPH_NODE_COUNT > 0

    print("Neo4j connection test:", ok)
    print("Node count:", GRAPH_NODE_COUNT)
    print("Labels:", labels)
    print("Relationships:", rels)

    if GRAPH_NODE_COUNT == 0:
        print("WARNING: Neo4j is running but the graph is empty. GraphRAG will likely use hybrid fallback.")

except Exception as e:
    GRAPH_DB_READY = False
    print("WARNING: Neo4j health check failed.")
    print(type(e).__name__, redact_sensitive(e))


Neo4j connection test: 1
Node count: 552
Labels: ['Document', 'Organization', 'Venue', 'Country', 'Region', 'ClimateTopic', 'ClimateRisk', 'Sector', 'Technology', 'Policy', 'Target', 'Finding', 'Indicator']
Relationships: ['PUBLISHED_BY', 'PUBLISHED_IN', 'DISCUSSES', 'MENTIONS_REGION', 'MENTIONS_TECHNOLOGY', 'DISCUSSES_POLICY', 'ADDRESSES', 'MENTIONS_COUNTRY', 'AFFECTS_SECTOR', 'DISCUSSES_RISK', 'LOCATED_IN', 'HAS_POLICY', 'HAS_RISK', 'REPORTS_INDICATOR', 'DISCUSSES_TARGET', 'SETS_TARGET', 'REPORTS_FINDING', 'SUPPORTED_BY', 'EVIDENCES_RISK', 'EVIDENCES_SECTOR', 'EVIDENCES_COUNTRY', 'EVIDENCES_TECHNOLOGY', 'IMPACTS', 'MITIGATES', 'OCCURS_IN']


## 8. Deep object extraction for real chunk IDs

This is the main fix for strict chunk-level matching.

The previous notebook only checked direct attributes like `c.chunk_id`. This version recursively searches nested dictionaries, dataclass fields, and object attributes inside each `BlendedChunk` so that `strict_chunk_recall@5`, `strict_chunk_ndcg@5`, and `strict_chunk_mrr@5` are based on the best available returned IDs.


In [8]:
def object_to_plain(obj, depth=0, max_depth=4):
    if depth > max_depth:
        return str(obj)

    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {str(k): object_to_plain(v, depth + 1, max_depth) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [object_to_plain(v, depth + 1, max_depth) for v in obj]

    if is_dataclass(obj):
        try:
            return object_to_plain(asdict(obj), depth + 1, max_depth)
        except Exception:
            pass

    if hasattr(obj, "__dict__"):
        try:
            return {
                str(k): object_to_plain(v, depth + 1, max_depth)
                for k, v in vars(obj).items()
                if not str(k).startswith("_")
            }
        except Exception:
            pass

    return str(obj)


def recursive_find_values(obj, key_patterns, value_filter=None):
    found = []

    def rec(x, path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                key = str(k).lower()
                new_path = f"{path}.{k}" if path else str(k)

                if any(p in key for p in key_patterns):
                    if isinstance(v, (str, int, float)):
                        val = str(v)
                        if value_filter is None or value_filter(val):
                            found.append((new_path, val))

                rec(v, new_path)

        elif isinstance(x, list):
            for i, item in enumerate(x):
                rec(item, f"{path}[{i}]")

    rec(obj)
    return found


def extract_chunk_item(c, retrieval_type):
    plain = object_to_plain(c)

    id_values = recursive_find_values(
        plain,
        key_patterns=["chunk_id", "chunkid", "doc_id", "document_id", "source_id", "id", "source"],
        value_filter=lambda v: len(v.strip()) > 0
    )

    text_values = recursive_find_values(
        plain,
        key_patterns=["text", "snippet", "content", "chunk_text", "page_content"],
        value_filter=lambda v: len(v.strip()) > 20
    )

    score_values = recursive_find_values(
        plain,
        key_patterns=["combined_score", "hybrid_score", "score"],
        value_filter=lambda v: True
    )

    page_values = recursive_find_values(
        plain,
        key_patterns=["page", "page_num", "page_number"],
        value_filter=lambda v: True
    )

    candidate_ids = []
    for _, val in id_values:
        if val not in candidate_ids:
            candidate_ids.append(val)

    # Prefer explicit chunk_id-like values.
    chunk_id = ""
    for _, val in id_values:
        if "chunk" in val.lower() or re.search(r"\b\d{4,}\b", val):
            chunk_id = val
            break
    if not chunk_id and candidate_ids:
        chunk_id = candidate_ids[0]

    doc_id = ""
    for path, val in id_values:
        if "doc" in path.lower() or "document" in path.lower():
            doc_id = val
            break

    text = text_values[0][1] if text_values else ""

    score = None
    for _, val in score_values:
        try:
            score = float(val)
            break
        except Exception:
            pass

    page = page_values[0][1] if page_values else ""

    return {
        "chunk_id": chunk_id,
        "doc_id": doc_id,
        "document_id": doc_id,
        "source": candidate_ids[0] if candidate_ids else "",
        "candidate_ids": candidate_ids,
        "page": page,
        "score": score,
        "combined_score": score,
        "text": text,
        "retrieval_type": retrieval_type,
        "raw_keys": list(plain.keys()) if isinstance(plain, dict) else [],
    }


def find_weight_targets(root_obj, max_depth=3):
    if root_obj is None:
        return []

    weight_names = {
        "bm25_weight", "bm25_w", "keyword_weight", "lexical_weight",
        "dense_weight", "dense_w", "semantic_weight", "embedding_weight",
        "alpha",
    }

    targets = []
    visited = set()

    def walk(obj, path, depth):
        if obj is None or depth > max_depth:
            return

        obj_id = id(obj)
        if obj_id in visited:
            return
        visited.add(obj_id)

        for name in weight_names:
            if hasattr(obj, name):
                try:
                    value = getattr(obj, name)
                    if isinstance(value, (int, float)):
                        targets.append((obj, path, name, value))
                except Exception:
                    pass

        if depth == max_depth or not hasattr(obj, "__dict__"):
            return

        for attr_name, child in vars(obj).items():
            if attr_name.startswith("_"):
                continue
            if isinstance(child, (str, int, float, bool, bytes, Path)):
                continue
            walk(child, f"{path}.{attr_name}", depth + 1)

    walk(root_obj, type(root_obj).__name__, 0)
    return targets


def apply_graphrag_weights(executor, bm25_weight: float, dense_weight: float):
    touched = []

    for obj, path, name, old_value in find_weight_targets(executor):
        try:
            if name in {"bm25_weight", "bm25_w", "keyword_weight", "lexical_weight"}:
                setattr(obj, name, float(bm25_weight))
                touched.append(f"{path}.{name}:{old_value}->{bm25_weight}")
            elif name in {"dense_weight", "dense_w", "semantic_weight", "embedding_weight"}:
                setattr(obj, name, float(dense_weight))
                touched.append(f"{path}.{name}:{old_value}->{dense_weight}")
            elif name == "alpha":
                setattr(obj, name, float(dense_weight))
                touched.append(f"{path}.{name}:{old_value}->{dense_weight}")
        except Exception:
            pass

    return sorted(set(touched))


def normalize_graphrag_result(result, measured_latency_ms: float):
    answer = getattr(result, "answer", "") or ""
    retrieval_type = getattr(result, "retrieval_type", "")
    latency_sec = getattr(result, "latency_sec", None)

    latency_ms = float(latency_sec) * 1000 if latency_sec is not None else measured_latency_ms

    blended_chunks = getattr(result, "blended_chunks", []) or []
    graph_hits = getattr(result, "graph_hits", []) or []
    citations = getattr(result, "citation_pages", []) or []

    results = [extract_chunk_item(c, retrieval_type=retrieval_type) for c in blended_chunks]

    return {
        "answer": answer,
        "results": results,
        "citations": citations,
        "latency_ms": latency_ms,
        "retrieval_type": retrieval_type,
        "template_used": getattr(result, "template_used", ""),
        "graph_hit_count": len(graph_hits),
        "blended_chunk_count": len(blended_chunks),
        "raw": result,
    }


def invoke_graphrag(query: str, top_k: int, bm25_weight: float, dense_weight: float, topic: str, graph_profile: str):
    if GRAPHRAG_EXECUTOR is None:
        raise RuntimeError("GraphRAG executor is not loaded.")

    touched_weights = apply_graphrag_weights(
        GRAPHRAG_EXECUTOR,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight,
    )

    start = time.perf_counter()
    result = GRAPHRAG_EXECUTOR.run(query)
    elapsed_ms = (time.perf_counter() - start) * 1000.0

    output = normalize_graphrag_result(result, elapsed_ms)
    output["weight_attributes_updated"] = touched_weights
    output["weight_control_status"] = "applied" if touched_weights else "not_exposed"

    return output


print("Visible weight targets:")
if GRAPHRAG_EXECUTOR is not None:
    targets = find_weight_targets(GRAPHRAG_EXECUTOR)
    for _, path, name, value in targets:
        print(f"- {path}.{name} = {value}")
    if not targets:
        print("No direct BM25/dense weight attributes found.")


Visible weight targets:
- GraphRAGExecutor.blender.hybrid_retriever.bm25_weight = 0.5


## 9. Load evaluation set and balanced sample

In [9]:
# -----------------------------
# D3 gold Q/A loading and validation
# -----------------------------

D3_GOLD_QA_FULL_PATH = ROOT / D3_GOLD_QA_PATH

def find_first_column(df: pd.DataFrame, candidates: list[str]):
    """Find a column by flexible normalized matching."""
    norm_to_original = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in df.columns
    }
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]", "", cand.lower())
        if key in norm_to_original:
            return norm_to_original[key]
    # substring fallback
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]", "", cand.lower())
        for norm, original in norm_to_original.items():
            if key and (key in norm or norm in key):
                return original
    return None


def parse_list_field(value):
    """Parse JSON/list/pipe/comma/semicolon values into a list of strings."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    if isinstance(value, tuple) or isinstance(value, set):
        return [str(x) for x in value if str(x).strip()]
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed if str(x).strip()]
        if isinstance(parsed, dict):
            return [str(x) for x in parsed.values() if str(x).strip()]
    except Exception:
        pass
    for sep in ["|", ";", ","]:
        if sep in text:
            return [x.strip() for x in text.split(sep) if x.strip()]
    return [text]


def parse_pages(value):
    pages = []
    for item in parse_list_field(value):
        for n in re.findall(r"\d+", str(item)):
            try:
                pages.append(int(n))
            except Exception:
                pass
    return sorted(set(pages))


def make_validation_table(validation_items):
    return pd.DataFrame(validation_items, columns=["check", "status", "details"])


validation_items = []

if not D3_GOLD_QA_FULL_PATH.exists():
    validation_items.append([
        "d3_gold_qa.csv exists",
        "FAIL",
        f"Missing required file: {D3_GOLD_QA_FULL_PATH}",
    ])
    validation_df = make_validation_table(validation_items)
    display(validation_df)
    raise FileNotFoundError(
        f"Required D3 gold Q/A file is missing: {D3_GOLD_QA_FULL_PATH}. "
        "Do not run final D3 metrics from the fallback D1 set."
    )

gold_df = pd.read_csv(D3_GOLD_QA_FULL_PATH)
validation_items.append(["d3_gold_qa.csv exists", "PASS", str(D3_GOLD_QA_FULL_PATH)])
validation_items.append(["row count", "PASS" if len(gold_df) > 0 else "FAIL", f"{len(gold_df)} rows"])
validation_items.append(["available columns", "INFO", ", ".join(map(str, gold_df.columns))])

query_col = find_first_column(gold_df, [
    # Prefer exact-retrieval wording when present. The human-readable question
    # remains in the `question` column for reports.
    "retrieval_query", "search_query", "query", "question", "Question",
    "prompt", "user_query", "input"
])
answer_col = find_first_column(gold_df, [
    "reference_answer", "answer", "gold_answer", "expected_answer", "ground_truth_answer", "target_answer"
])
topic_col = find_first_column(gold_df, [
    "true_topic", "topic", "expected_topic", "label", "category", "climate_topic"
])
chunk_col = find_first_column(gold_df, [
    "relevant_ids", "relevant_chunk_ids", "gold_chunk_ids", "chunk_ids",
    "supporting_chunks", "supporting_chunk_ids", "citation_chunk_ids"
])
doc_col = find_first_column(gold_df, [
    "relevant_doc_ids", "gold_doc_ids", "document_id", "doc_id", "source_doc",
    "paper_id", "file_id", "filename", "file_name", "source"
])
page_col = find_first_column(gold_df, [
    "relevant_pages", "gold_pages", "page", "page_number", "page_num",
    "supporting_pages", "citation_pages"
])

column_checks = {
    "query column": query_col,
    "reference answer column": answer_col,
    "true topic column (optional for accuracy)": topic_col,
    "gold chunk IDs column": chunk_col,
    "gold document IDs column": doc_col,
    "gold page column": page_col,
}

for check, col in column_checks.items():
    if col:
        validation_items.append([check, "PASS", col])
    elif check.startswith("true topic"):
        validation_items.append([check, "WARN", "Missing; topic accuracy will be N/A unless topic is inferred for routing only."])
    else:
        validation_items.append([check, "WARN", "Missing"])

critical_missing = []
if query_col is None:
    critical_missing.append("query/question")
if answer_col is None:
    critical_missing.append("reference_answer/answer")
if chunk_col is None and doc_col is None and page_col is None:
    critical_missing.append("at least one relevance column: chunk IDs, document IDs, or pages")

validation_status = "PASS" if not critical_missing else "FAIL"
validation_items.append([
    "D3 gold schema usable",
    validation_status,
    "OK" if not critical_missing else "Missing: " + ", ".join(critical_missing),
])

validation_df = make_validation_table(validation_items)
display(validation_df)

if critical_missing:
    raise ValueError(
        "data/gold/d3_gold_qa.csv is present but incomplete. Missing: "
        + ", ".join(critical_missing)
        + ". Do not fabricate final metrics."
    )

records_eval = []

for i, item in gold_df.iterrows():
    query = normalize_text(item.get(query_col, ""))
    reference_answer = normalize_text(item.get(answer_col, ""))

    raw_topic = item.get(topic_col, None) if topic_col else None
    gold_topic = normalize_topic(raw_topic)

    if gold_topic == "unknown":
        inferred_topic = infer_true_topic({"query": query, "reference_answer": reference_answer})
        topic_for_routing = inferred_topic
        topic_accuracy_eligible = False
        true_topic_source = "inferred_for_routing_only"
    else:
        topic_for_routing = gold_topic
        topic_accuracy_eligible = True
        true_topic_source = "gold"

    relevant_chunk_ids = parse_list_field(item.get(chunk_col, None)) if chunk_col else []
    relevant_doc_ids = parse_list_field(item.get(doc_col, None)) if doc_col else []
    relevant_pages = parse_pages(item.get(page_col, None)) if page_col else []

    records_eval.append({
        "index": int(i),
        "query": query,
        "reference_answer": reference_answer,
        "true_topic": gold_topic if gold_topic != "unknown" else pd.NA,
        "topic_for_routing": topic_for_routing,
        "true_topic_source": true_topic_source,
        "topic_accuracy_eligible": bool(topic_accuracy_eligible),
        "relevant_ids": relevant_chunk_ids,
        "relevant_chunk_ids": relevant_chunk_ids,
        "relevant_document_ids": relevant_doc_ids,
        "relevant_pages": relevant_pages,
        "eval_source": "d3_gold_qa.csv",
    })

eval_df = pd.DataFrame(records_eval)
eval_df = eval_df[eval_df["query"].str.len() > 0].reset_index(drop=True)

if eval_df.empty:
    raise ValueError("d3_gold_qa.csv loaded but no usable query rows were parsed.")

if RUN_MODE == "smoke":
    eval_run_df = eval_df.head(MAX_EVAL_ROWS).copy()
elif RUN_MODE == "final":
    eval_run_df = eval_df.head(MAX_EVAL_ROWS).copy()
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'final'.")

evaluation_status = (
    "final_d3_gold_run"
    if RUN_MODE == "final" and len(eval_run_df) >= 10
    else "smoke_d3_gold_run"
)

topic_accuracy_basis = (
    "gold_true_topic_only"
    if bool(eval_run_df["topic_accuracy_eligible"].any())
    else "not_available_true_topic_missing"
)

print("Evaluation source:", D3_GOLD_QA_FULL_PATH)
print("RUN_MODE:", RUN_MODE)
print("MAX_EVAL_ROWS:", MAX_EVAL_ROWS)
print("Selected rows:", len(eval_run_df))
print("Evaluation status:", evaluation_status)
print("Topic accuracy basis:", topic_accuracy_basis)
print("Rows with gold true_topic:", int(eval_run_df["topic_accuracy_eligible"].sum()))
print("Rows with inferred topic for routing only:", int((~eval_run_df["topic_accuracy_eligible"]).sum()))

display(eval_run_df[[
    "index", "query", "true_topic", "topic_for_routing", "true_topic_source",
    "topic_accuracy_eligible", "relevant_chunk_ids", "relevant_document_ids", "relevant_pages"
]].head(20))


,check,status,details
0,d3_gold_qa.csv exists,PASS,D:\BUID\Year 4 2025 - 2026\third semester\Spec...
1,row count,PASS,15 rows
2,available columns,INFO,"query_id, question_id, question, query, retrie..."
3,query column,PASS,retrieval_query
4,reference answer column,PASS,reference_answer
5,true topic column (optional for accuracy),PASS,true_topic
6,gold chunk IDs column,PASS,relevant_chunk_ids
7,gold document IDs column,PASS,relevant_doc_id
8,gold page column,PASS,gold_page_start
9,D3 gold schema usable,PASS,OK


Evaluation source: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\data\gold\d3_gold_qa.csv
RUN_MODE: final
MAX_EVAL_ROWS: 15
Selected rows: 15
Evaluation status: final_d3_gold_run
Topic accuracy basis: gold_true_topic_only
Rows with gold true_topic: 15
Rows with inferred topic for routing only: 0


,index,query,true_topic,topic_for_routing,true_topic_source,topic_accuracy_eligible,relevant_chunk_ids,relevant_document_ids,relevant_pages
0,0,FeederBW low-voltage Netze BW one-minute feed-...,technology_innovation,technology_innovation,gold,True,[chunk_036782],[treutlein_2026_real_world_energy_data_200_fee...,[1]
1,1,cereal crops phenology sowing date thermal tre...,adaptation,adaptation,gold,True,[chunk_006699],[fatima_2020_fingerprints_climate_warming_cere...,[11]
2,2,interactive photogrammetric dendrometry uncali...,adaptation,adaptation,gold,True,[chunk_036179],[ravi_2025_citizen_centered_climate_intelligen...,[7]
3,3,Massachusetts substations transformer thermal ...,climate_science,climate_science,gold,True,[chunk_023284],[shah_2025_assessing_climate_vulnerability_ris...,[3]
4,4,energy advisors accused commercial interests j...,policy_governance,policy_governance,gold,True,[chunk_039454],[scheller_2021_stakeholder_dynamics_residentia...,[19]
5,5,offertilisers explosives ammoniahasbeen geo-po...,technology_innovation,technology_innovation,gold,True,[chunk_035742],[smith_2019_current_future_role_haber_bosch_am...,[2]
6,6,resilience conflate structure function socio-t...,adaptation,adaptation,gold,True,[chunk_021253],[smith_2010_politics_social_ecological_resilie...,[4]
7,7,climate warming cereal crops phenological phas...,climate_science,climate_science,gold,True,[chunk_006718],[fatima_2020_fingerprints_climate_warming_cere...,[12]
8,8,halving bullet liquefied silver raising underg...,mitigation,mitigation,gold,True,[chunk_033119],[balcombe_2019_how_decarbonise_international_s...,[4]
9,9,AMOC rate-induced oscillatory bifurcations fre...,climate_science,climate_science,gold,True,[chunk_036501],[alkhayuon_2019_basin_bifurcations_oscillatory...,[17]


## 10. Strict, document/page, and diagnostic metrics

The official retrieval metrics are separated clearly:

- `strict_chunk_recall@5`, `strict_chunk_ndcg@5`, `strict_chunk_mrr@5`
- `document_recall@5`, `document_ndcg@5`, `document_mrr@5`
- `page_window_recall@5`

`soft_Recall@5` is diagnostic only. It uses lexical overlap to explain related evidence, but it should not be presented as gold relevance.


In [10]:
# -----------------------------
# Metric helpers
# -----------------------------

WORD_RE = re.compile(r"[A-Za-z0-9_]{3,}")

def normalize_id(value: Any) -> str:
    return re.sub(r"[^a-z0-9]", "", normalize_text(value).lower())


# Load corpus chunks for evaluation-level ID expansion.
# GraphRAG outputs often expose only chunk_id. For document/page metrics we must map
# chunk_id -> document_id/page_start/page_end; otherwise document/page metrics become
# false zeros even when the retrieved chunk comes from the correct source document.
CHUNK_LOOKUP_BY_ID = {}
CHUNK_LOOKUP_BY_NORMALIZED_ID = {}
try:
    _chunks_for_eval = json.loads((ROOT / "data" / "sample" / "sample_chunks.json").read_text(encoding="utf-8"))
    CHUNK_LOOKUP_BY_ID = {str(c.get("chunk_id")): c for c in _chunks_for_eval if c.get("chunk_id")}
    CHUNK_LOOKUP_BY_NORMALIZED_ID = {normalize_id(k): v for k, v in CHUNK_LOOKUP_BY_ID.items()}
    print("Loaded chunk lookup for metric expansion:", len(CHUNK_LOOKUP_BY_ID))
except Exception as e:
    print("WARNING: could not load chunk lookup for document/page metric expansion:", redact_sensitive(e))


def tokens(text: Any) -> set[str]:
    return set(WORD_RE.findall(normalize_text(text).lower()))


def lexical_overlap(a: Any, b: Any) -> float:
    a_toks = tokens(a)
    b_toks = tokens(b)
    if not a_toks or not b_toks:
        return 0.0
    return len(a_toks & b_toks) / max(1, min(len(a_toks), len(b_toks)))


def id_tail(value: Any, n: int = 6) -> str:
    digits = "".join(re.findall(r"\d+", normalize_text(value)))
    return digits[-n:] if digits else ""


def resolve_chunk_from_any_id(value: Any) -> dict | None:
    text = normalize_text(value)
    if not text:
        return None
    if text in CHUNK_LOOKUP_BY_ID:
        return CHUNK_LOOKUP_BY_ID[text]
    norm = normalize_id(text)
    if norm in CHUNK_LOOKUP_BY_NORMALIZED_ID:
        return CHUNK_LOOKUP_BY_NORMALIZED_ID[norm]
    return None


def get_result_text(result: Any) -> str:
    if isinstance(result, str):
        return result
    if not isinstance(result, dict):
        return str(result)
    for key in ["text", "content", "chunk_text", "page_content", "snippet"]:
        if result.get(key):
            return normalize_text(result.get(key))
    # If the result only contains a chunk id, recover the chunk text from the corpus.
    for rid in get_result_ids(result):
        chunk = resolve_chunk_from_any_id(rid)
        if chunk and chunk.get("text"):
            return normalize_text(chunk.get("text"))
    return ""


def get_result_ids(result: Any) -> set[str]:
    ids = set()
    if isinstance(result, dict):
        for key in [
            "chunk_id", "doc_id", "document_id", "source", "source_id",
            "id", "filename", "file_name"
        ]:
            if result.get(key):
                ids.add(str(result.get(key)))
        for val in result.get("candidate_ids", []) or []:
            if val:
                ids.add(str(val))
    return ids


def get_result_document_ids(result: Any) -> set[str]:
    docs = set()
    if isinstance(result, dict):
        for key in ["doc_id", "document_id", "source_document_id"]:
            if result.get(key):
                docs.add(str(result.get(key)))
    for rid in get_result_ids(result):
        chunk = resolve_chunk_from_any_id(rid)
        if chunk and chunk.get("document_id"):
            docs.add(str(chunk.get("document_id")))
    return docs


def get_result_pages(result: Any) -> set[int]:
    pages = set()
    if isinstance(result, dict):
        for key in ["page", "page_number", "page_num", "page_start", "page_end"]:
            value = result.get(key)
            for n in re.findall(r"\d+", normalize_text(value)):
                try:
                    pages.add(int(n))
                except Exception:
                    pass
    # If the result only contains a chunk id, recover page_start/page_end from the corpus.
    for rid in get_result_ids(result):
        chunk = resolve_chunk_from_any_id(rid)
        if chunk:
            for key in ["page_start", "page_end"]:
                try:
                    if chunk.get(key) not in [None, ""]:
                        pages.add(int(chunk.get(key)))
                except Exception:
                    pass
    return pages


def strict_chunk_match(result: Any, row: dict) -> bool:
    gold_ids = [str(x) for x in row.get("relevant_chunk_ids", row.get("relevant_ids", []))]
    if not gold_ids:
        return False

    result_ids = get_result_ids(result)

    gold_norm = {normalize_id(x) for x in gold_ids if normalize_id(x)}
    result_norm = {normalize_id(x) for x in result_ids if normalize_id(x)}

    if gold_norm & result_norm:
        return True

    gold_tail = {id_tail(x) for x in gold_ids if id_tail(x)}
    result_tail = {id_tail(x) for x in result_ids if id_tail(x)}

    return bool(gold_tail and result_tail and (gold_tail & result_tail))


def document_match(result: Any, row: dict) -> bool:
    gold_doc_ids = [str(x) for x in row.get("relevant_document_ids", [])]
    if not gold_doc_ids:
        return False

    result_ids = get_result_ids(result)
    result_doc_ids = get_result_document_ids(result)
    result_text = get_result_text(result)

    gold_norm = {normalize_id(x) for x in gold_doc_ids if normalize_id(x)}
    result_norm = {normalize_id(x) for x in result_ids if normalize_id(x)}
    result_doc_norm = {normalize_id(x) for x in result_doc_ids if normalize_id(x)}

    if gold_norm & result_doc_norm:
        return True

    if gold_norm & result_norm:
        return True

    text_norm = normalize_id(result_text)
    return any(g and g in text_norm for g in gold_norm)


def page_window_match(result: Any, row: dict, window: int = 1) -> bool:
    gold_pages = row.get("relevant_pages", []) or []
    if not gold_pages:
        return False

    result_pages = get_result_pages(result)
    if not result_pages:
        return False

    # If gold document IDs are available and the result exposes document IDs,
    # require document match as well. Otherwise evaluate the page window only.
    result_ids_available = bool(get_result_ids(result))
    if row.get("relevant_document_ids") and result_ids_available and not document_match(result, row):
        return False

    for gp in gold_pages:
        for rp in result_pages:
            if abs(int(gp) - int(rp)) <= window:
                return True
    return False


def soft_relevance_score(result: Any, row: dict) -> float:
    """
    Diagnostic only. This is not gold relevance.
    It checks lexical overlap with the reference answer/query to explain why a non-exact chunk
    may still look semantically related.
    """
    if strict_chunk_match(result, row):
        return 1.0
    result_text = get_result_text(result)
    reference = row.get("reference_answer", "")
    query = row.get("query", "")
    ref_overlap = lexical_overlap(reference, result_text)
    query_overlap = lexical_overlap(query, result_text)
    return round(max(ref_overlap, query_overlap * 0.5), 4)


def recall_at_k(results: list[dict], row: dict, match_fn, k: int = 5) -> float:
    return float(any(match_fn(r, row) for r in results[:k]))


def mrr_at_k(results: list[dict], row: dict, match_fn, k: int = 5) -> float:
    for rank, result in enumerate(results[:k], start=1):
        if match_fn(result, row):
            return 1.0 / rank
    return 0.0


def ndcg_at_k(results: list[dict], row: dict, match_fn, k: int = 5) -> float:
    gains = [1.0 if match_fn(r, row) else 0.0 for r in results[:k]]
    dcg = sum(gain / math.log2(i + 2) for i, gain in enumerate(gains))
    ideal = sorted(gains, reverse=True)
    idcg = sum(gain / math.log2(i + 2) for i, gain in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def soft_recall_at_k(results: list[dict], row: dict, k: int = 5, threshold: float = 0.10) -> float:
    return float(any(soft_relevance_score(r, row) >= threshold for r in results[:k]))


def citation_correctness_score(output: dict, row: dict, results: list[dict]) -> float:
    citations = output.get("citations") or []
    if not citations:
        return 0.0

    # Strongest evidence: cited/retrieved evidence matches page-window, document, or strict chunk.
    if any(page_window_match(r, row) or document_match(r, row) or strict_chunk_match(r, row) for r in results[:5]):
        return 1.0

    return 0.0


def citation_hit_rate(output: dict, row: dict, results: list[dict]) -> float:
    citations = output.get("citations") or []
    if not citations:
        return 0.0

    # Hit rate only checks whether the answer had citations attached.
    # Correctness is handled separately by citation_correctness_score().
    return 1.0


def faithfulness_score(output: dict, answer: str, results: list[dict]) -> float:
    context_text = " ".join(get_result_text(r) for r in results[:5])
    if not answer.strip() or not context_text.strip():
        return 0.0

    answer_toks = tokens(answer)
    context_toks = tokens(context_text)
    if not answer_toks:
        return 0.0

    return round(len(answer_toks & context_toks) / max(1, len(answer_toks)), 4)


def answer_relevance_score(output: dict, answer: str, row: dict) -> float:
    query_overlap = lexical_overlap(row.get("query", ""), answer)
    reference_overlap = lexical_overlap(row.get("reference_answer", ""), answer)

    if row.get("reference_answer"):
        return round((query_overlap + reference_overlap) / 2.0, 4)
    return round(query_overlap, 4)


def composite_score(metric_row: dict) -> float:
    """
    Composite is used only for helps/hurts, not as a headline metric.
    It emphasizes fair GraphRAG evidence quality rather than only exact chunk matching.
    """
    return (
        metric_row["strict_chunk_recall@5"]
        + metric_row["document_recall@5"]
        + metric_row["document_ndcg@5"]
        + metric_row["document_mrr@5"]
        + metric_row["page_window_recall@5"]
        + metric_row["faithfulness"]
        + metric_row["answer_relevance"]
        + metric_row["citation_correctness"]
    )


def result_signature(results: list[dict]) -> str:
    ids = []
    for r in results[:5]:
        rid = r.get("chunk_id") or "|".join((r.get("candidate_ids") or [])[:2])
        ids.append(normalize_id(rid)[:30])
    return " > ".join(ids)


# Backward-compatible helper name used in the smoke-test debug table.
def strict_id_match(result: Any, row: dict) -> bool:
    return strict_chunk_match(result, row)


print("Metric helpers loaded:")
print("- strict_chunk_recall@5 / strict_chunk_ndcg@5 / strict_chunk_mrr@5")
print("- document_recall@5 / document_ndcg@5 / document_mrr@5")
print("- page_window_recall@5")
print("- soft_Recall@5 is diagnostic only, not gold relevance")


Loaded chunk lookup for metric expansion: 49541
Metric helpers loaded:
- strict_chunk_recall@5 / strict_chunk_ndcg@5 / strict_chunk_mrr@5
- document_recall@5 / document_ndcg@5 / document_mrr@5
- page_window_recall@5
- soft_Recall@5 is diagnostic only, not gold relevance


In [11]:
# Sanity check: metric helpers must be loaded before the smoke test.
required_metric_helpers = [
    "strict_chunk_match",
    "document_match",
    "page_window_match",
    "soft_relevance_score",
    "strict_id_match",
]
missing_helpers = [name for name in required_metric_helpers if name not in globals()]
if missing_helpers:
    raise NameError(f"Missing metric helper functions: {missing_helpers}. Run the metric helper cell first.")
print("Metric helper sanity check passed.")


Metric helper sanity check passed.


## 11. Gemini-only model fallback

In [12]:
GEMINI_RESPONSE_CACHE = {}


def list_supported_gemini_models():
    """
    Ask Gemini API which models this API key can actually use for generateContent.
    This avoids hard-coding deprecated or unavailable model names.
    """
    gemini_api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not gemini_api_key:
        raise RuntimeError("GEMINI_API_KEY is missing.")

    url = f"https://generativelanguage.googleapis.com/v1beta/models?key={gemini_api_key}"
    response = requests.get(url, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(
            f"Could not list Gemini models. HTTP {response.status_code}: {redact_sensitive(response.text[:500])}"
        )

    data = response.json()
    models = data.get("models", [])

    supported = []
    for model in models:
        methods = model.get("supportedGenerationMethods", [])
        if "generateContent" not in methods:
            continue

        name = model.get("name", "")
        base = model.get("baseModelId", "")

        if name.startswith("models/"):
            name = name.replace("models/", "", 1)

        # Prefer baseModelId when available because generateContent endpoint accepts it.
        model_id = base or name

        if model_id and model_id not in supported:
            supported.append(model_id)

    return supported


available_gemini_models = list_supported_gemini_models()
print("Gemini models available for generateContent with this key:")
for m in available_gemini_models:
    print("-", m)

# Keep preferred models only if they are available.
GEMINI_MODEL_FALLBACKS = [
    m for m in PREFERRED_GEMINI_MODELS
    if m in available_gemini_models
]

# If none of the preferred models are available, use available Flash-like models first.
if not GEMINI_MODEL_FALLBACKS:
    flash_like = [
        m for m in available_gemini_models
        if "flash" in m.lower()
        and "image" not in m.lower()
        and "audio" not in m.lower()
        and "tts" not in m.lower()
    ]
    GEMINI_MODEL_FALLBACKS = flash_like[:3]

if not GEMINI_MODEL_FALLBACKS:
    raise RuntimeError(
        "No Gemini generateContent models are available for this key. "
        "Check Google AI Studio API access."
    )

print("Gemini fallback order actually used:")
for m in GEMINI_MODEL_FALLBACKS:
    print("-", m)


def gemini_call_with_model_fallback(self, prompt):
    gemini_api_key = (
        getattr(self, "gemini_api_key", None)
        or os.environ.get("GEMINI_API_KEY")
        or os.environ.get("GOOGLE_API_KEY")
    )

    if not gemini_api_key:
        raise RuntimeError("GEMINI_API_KEY is missing.")

    prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    if prompt_hash in GEMINI_RESPONSE_CACHE:
        return GEMINI_RESPONSE_CACHE[prompt_hash]

    last_status = None
    last_message = None

    for model in GEMINI_MODEL_FALLBACKS:
        url = (
            "https://generativelanguage.googleapis.com/v1beta/models/"
            f"{model}:generateContent?key={gemini_api_key}"
        )

        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": 0.2,
                "maxOutputTokens": 700,
            },
        }

        for attempt in range(1, GEMINI_RETRIES_PER_MODEL + 1):
            response = requests.post(url, json=payload, timeout=60)
            last_status = response.status_code

            if response.status_code == 200:
                data = response.json()
                answer = data["candidates"][0]["content"]["parts"][0]["text"]
                GEMINI_RESPONSE_CACHE[prompt_hash] = answer
                print(f"Gemini answered with model: {model}")
                return answer

            retry_after = response.headers.get("Retry-After")
            wait_time = int(retry_after) if retry_after and str(retry_after).isdigit() else GEMINI_BASE_WAIT_SECONDS * attempt

            try:
                last_message = redact_sensitive(response.json().get("error", {}).get("message", response.text[:300]))
            except Exception:
                last_message = redact_sensitive(response.text[:300])

            if response.status_code in {429, 503}:
                print(
                    f"Gemini {model} returned HTTP {response.status_code}. "
                    f"Waiting {wait_time}s before trying again/next model..."
                )
                time.sleep(wait_time)
                continue

            if response.status_code in {403, 404}:
                # 404 means this model is not available for this key/API/version.
                # Do not crash; try the next listed model.
                print(f"Gemini {model} returned HTTP {response.status_code}; trying next available model.")
                break

            raise RuntimeError(
                f"Gemini request failed with HTTP {response.status_code}. "
                f"Message: {last_message}"
            )

    raise RuntimeError(
        "Gemini quota/model fallback failed for this run. "
        f"Last HTTP status: {last_status}. Last message: {last_message}. "
        "This is usually quota/rate-limit pressure, not a notebook logic error. "
        "Wait for quota reset, reduce MAX_EVAL_ROWS, or use a key/project with higher quota."
    )


def restore_generator_generate_if_needed():
    generator = getattr(GRAPHRAG_EXECUTOR, "generator", None)
    if generator is None:
        return
    cls_generate = getattr(type(generator), "generate", None)
    if cls_generate is not None:
        generator.generate = types.MethodType(cls_generate, generator)


def patch_all_gemini_callers():
    if GRAPHRAG_EXECUTOR is None:
        raise RuntimeError("GraphRAG executor is not loaded.")

    restore_generator_generate_if_needed()

    patched = []
    objects_to_scan = [GRAPHRAG_EXECUTOR]
    if hasattr(GRAPHRAG_EXECUTOR, "__dict__"):
        objects_to_scan.extend(list(vars(GRAPHRAG_EXECUTOR).values()))

    for obj in objects_to_scan:
        if obj is None:
            continue
        if hasattr(obj, "_call_gemini"):
            obj._call_gemini = types.MethodType(gemini_call_with_model_fallback, obj)
            patched.append(f"{type(obj).__name__}._call_gemini")

    print("Patched Gemini callers:", sorted(set(patched)))
    print("Gemini fallback order:", GEMINI_MODEL_FALLBACKS)


patch_all_gemini_callers()


Gemini models available for generateContent with this key:
- gemini-2.5-flash
- gemini-2.5-pro
- gemini-2.0-flash
- gemini-2.0-flash-001
- gemini-2.0-flash-lite-001
- gemini-2.0-flash-lite
- gemini-2.5-flash-preview-tts
- gemini-2.5-pro-preview-tts
- gemma-4-26b-a4b-it
- gemma-4-31b-it
- gemini-flash-latest
- gemini-flash-lite-latest
- gemini-pro-latest
- gemini-2.5-flash-lite
- gemini-2.5-flash-image
- gemini-3-pro-preview
- gemini-3-flash-preview
- gemini-3.1-pro-preview
- gemini-3.1-pro-preview-customtools
- gemini-3.1-flash-lite-preview
- gemini-3.1-flash-lite
- gemini-3-pro-image-preview
- gemini-3-pro-image
- nano-banana-pro-preview
- gemini-3.1-flash-image-preview
- gemini-3.1-flash-image
- gemini-3.5-flash
- lyria-3-clip-preview
- lyria-3-pro-preview
- gemini-3.1-flash-tts-preview
- gemini-robotics-er-1.5-preview
- gemini-robotics-er-1.6-preview
- gemini-2.5-computer-use-preview-10-2025
- antigravity-preview-05-2026
- deep-research-max-preview-04-2026
- deep-research-preview-04

## 12. Smoke test and ID extraction check

The key part is the displayed `candidate_ids`. If these include the same style as the gold IDs, strict metrics can work.


In [13]:
if not USING_GRAPHRAG_EXECUTOR:
    raise RuntimeError("Stop here: USING_GRAPHRAG_EXECUTOR is False.")

test_row = eval_run_df.iloc[0].to_dict()
test_query = test_row["query"]

test_out = invoke_graphrag(
    query=test_query,
    top_k=5,
    bm25_weight=0.50,
    dense_weight=0.50,
    topic="none",
    graph_profile="balanced_evidence",
)

print("Returned chunks:", len(test_out["results"]))
print("Retrieval type:", test_out.get("retrieval_type"))
print("Template used:", test_out.get("template_used"))
print("Gold relevant IDs:", test_row["relevant_ids"][:5])

debug_rows = []
for rank, r in enumerate(test_out["results"][:5], start=1):
    debug_rows.append({
        "rank": rank,
        "chunk_id": r.get("chunk_id"),
        "candidate_ids": r.get("candidate_ids"),
        "score": r.get("score"),
        "strict_match": strict_id_match(r, test_row),
        "soft_score": soft_relevance_score(r, test_row),
        "text_preview": get_result_text(r)[:180],
    })

display(pd.DataFrame(debug_rows))

print("Answer preview:")
print(test_out["answer"][:500])


Gemini answered with model: gemini-3.1-flash-lite
Returned chunks: 16
Retrieval type: graph_guided
Template used: graphrag_finding_document_grounding
Gold relevant IDs: ['chunk_036782']


,rank,chunk_id,candidate_ids,score,strict_match,soft_score,text_preview
0,1,chunk_036782,"[chunk_036782, treutlein_2026_real_world_energ...",1.000000,True,1.0000,"rablechallengesinoperationandplanning,research..."
1,2,chunk_036783,"[chunk_036783, treutlein_2026_real_world_energ...",0.502568,False,0.4737,photovoltaic feed-in and one-minute temporal r...
2,3,chunk_036851,"[chunk_036851, treutlein_2026_real_world_energ...",0.426344,False,0.2368,(a) Feeder F with 102 housing units and no oth...
3,4,chunk_036846,"[chunk_036846, treutlein_2026_real_world_energ...",0.426056,False,0.2632,to the nearest whole number. Consumption of in...
4,5,chunk_036814,"[chunk_036814, treutlein_2026_real_world_energ...",0.419743,False,0.3158,assigned to a single LV feeder rather than mul...


Answer preview:
The FeederBW dataset, provided by the German distribution system operator Netze BW, contains real-world energy data from 200 low-voltage feeders recorded between 2023 and 2025 (Treutlein et al., 2026, p. 1). This dataset is distinguished by its one-minute temporal resolution and its inclusion of high photovoltaic feed-in measurements (Treutlein et al., 2026, p. 1). Additionally, the data incorporates weather information and detailed metadata, such as the number of housing units and the installed


In [14]:
GRAPHRAG_EXECUTOR.selector.llm_classify = False
GRAPHRAG_EXECUTOR.selector.gemini_api_key = ""
GRAPHRAG_EXECUTOR.selector.classify_intent = lambda query: GRAPHRAG_EXECUTOR.selector._classify_with_keywords(query)

print("classifier disabled:", GRAPHRAG_EXECUTOR.selector.llm_classify)
print("selector Gemini key cleared:", GRAPHRAG_EXECUTOR.selector.gemini_api_key == "")

classifier disabled: False
selector Gemini key cleared: True


## 13. Prequential comparison loop

In [15]:
if not USING_GRAPHRAG_EXECUTOR:
    raise RuntimeError("USING_GRAPHRAG_EXECUTOR is False. Do not run the comparison loop yet.")

if "eval_run_df" not in globals() or eval_run_df.empty:
    raise RuntimeError("eval_run_df is missing or empty. Run the D3 gold Q/A loading cell first.")

records = []
feedback_events = []

if RESUME_FROM_CHECKPOINT and PER_QUERY_PATH.exists():
    old_per_query = pd.read_csv(PER_QUERY_PATH)
    allowed_indices = set(eval_run_df["index"].astype(int).tolist())
    old_per_query = old_per_query[old_per_query["index"].astype(int).isin(allowed_indices)].copy()
    records = old_per_query.to_dict("records")
    print("Loaded checkpoint rows:", len(records))

if RESUME_FROM_CHECKPOINT and FEEDBACK_EVENTS_PATH.exists():
    old_feedback = pd.read_csv(FEEDBACK_EVENTS_PATH)
    allowed_indices = set(eval_run_df["index"].astype(int).tolist())
    old_feedback = old_feedback[old_feedback["index"].astype(int).isin(allowed_indices)].copy()
    feedback_events = old_feedback.to_dict("records")

completed = {(int(r["index"]), r["method"]) for r in records if "index" in r and "method" in r}

print(f"RUN_MODE={RUN_MODE}")
print(f"Running up to {len(eval_run_df)} D3 gold queries × 3 methods = {len(eval_run_df) * 3} GraphRAG calls")
print("Already completed calls:", len(completed))


def save_checkpoints():
    pd.DataFrame(records).to_csv(PER_QUERY_PATH, index=False)
    pd.DataFrame(feedback_events).to_csv(FEEDBACK_EVENTS_PATH, index=False)


start_all = time.perf_counter()

for row_i, row in enumerate(eval_run_df.to_dict("records"), start=1):
    idx = int(row["index"])
    query = row["query"]

    topic_for_routing = normalize_topic(row.get("topic_for_routing", "unknown"))
    gold_true_topic = normalize_topic(row.get("true_topic", "unknown"))
    topic_accuracy_eligible = bool(row.get("topic_accuracy_eligible", False))

    predicted_topic, prediction_source = predict_topic_with_fallback(query)
    routing_topic = predicted_topic if predicted_topic != "unknown" else topic_for_routing

    if topic_accuracy_eligible and gold_true_topic != "unknown":
        topic_correct = int(predicted_topic == gold_true_topic)
    else:
        topic_correct = pd.NA

    gated_weights = TOPIC_ROUTING_PROFILES.get(routing_topic, STATIC_WEIGHTS)
    adaptive_weights = feedback_adapter.get_weights(routing_topic)

    methods = {
        "static_graphrag": {
            "topic_source": "none",
            "routing_topic": "none",
            "bm25_weight": STATIC_WEIGHTS["bm25_weight"],
            "dense_weight": STATIC_WEIGHTS["dense_weight"],
            "graph_profile": "balanced_evidence",
            "adaptation_target": "none",
        },
        "topic_gated_graphrag": {
            "topic_source": prediction_source,
            "routing_topic": routing_topic,
            "bm25_weight": gated_weights["bm25_weight"],
            "dense_weight": gated_weights["dense_weight"],
            "graph_profile": GRAPH_PROFILES.get(routing_topic, "balanced_evidence"),
            "adaptation_target": "BM25/dense weights + graph profile signal",
        },
        "feedback_adaptive_graphrag": {
            "topic_source": prediction_source + " + FeedbackAdapter",
            "routing_topic": routing_topic,
            "bm25_weight": adaptive_weights["bm25_weight"],
            "dense_weight": adaptive_weights["dense_weight"],
            "graph_profile": GRAPH_PROFILES.get(routing_topic, "balanced_evidence"),
            "adaptation_target": "BM25/dense weights; graph profile logged if supported",
        },
    }

    print(f"\nQuery {row_i}/{len(eval_run_df)} | gold_topic={gold_true_topic} | predicted={predicted_topic} ({prediction_source}) | routing={routing_topic}")
    print("Query:", query[:160])

    query_method_rows = {r["method"]: r for r in records if int(r.get("index", -1)) == idx}

    for method_name, cfg in methods.items():
        if (idx, method_name) in completed:
            print(f"  Skipping already completed: {method_name}")
            continue

        print(f"  Running: {method_name} ...", end=" ")

        try:
            output = invoke_graphrag(
                query=query,
                top_k=TOP_K,
                bm25_weight=cfg["bm25_weight"],
                dense_weight=cfg["dense_weight"],
                topic=cfg["routing_topic"],
                graph_profile=cfg["graph_profile"],
            )
            time.sleep(SLEEP_BETWEEN_GRAPH_RAG_CALLS_SEC)
        except Exception as e:
            save_checkpoints()
            print("FAILED. Checkpoint saved.")
            print("Error type:", type(e).__name__)
            print("Error message:", redact_sensitive(e))
            raise

        print("done")

        results = output["results"][:TOP_K]
        answer = normalize_text(output.get("answer", ""))

        metric_row = {
            "index": idx,
            "query": query,
            "reference_answer": row.get("reference_answer", ""),
            "true_topic": gold_true_topic if gold_true_topic != "unknown" else pd.NA,
            "topic_for_routing": topic_for_routing,
            "true_topic_source": row.get("true_topic_source", "unknown"),
            "topic_accuracy_eligible": topic_accuracy_eligible,
            "predicted_topic": predicted_topic,
            "prediction_source": prediction_source,
            "topic_correct": topic_correct,
            "method": method_name,
            "topic_source": cfg["topic_source"],
            "routing_topic": cfg["routing_topic"],
            "bm25_weight": cfg["bm25_weight"],
            "dense_weight": cfg["dense_weight"],
            "graph_profile": cfg["graph_profile"],
            "adaptation_target": cfg["adaptation_target"],

            # Required separated retrieval metrics
            "strict_chunk_recall@5": recall_at_k(results, row, strict_chunk_match, TOP_K),
            "strict_chunk_ndcg@5": ndcg_at_k(results, row, strict_chunk_match, TOP_K),
            "strict_chunk_mrr@5": mrr_at_k(results, row, strict_chunk_match, TOP_K),
            "document_recall@5": recall_at_k(results, row, document_match, TOP_K),
            "document_ndcg@5": ndcg_at_k(results, row, document_match, TOP_K),
            "document_mrr@5": mrr_at_k(results, row, document_match, TOP_K),
            "page_window_recall@5": recall_at_k(results, row, page_window_match, TOP_K),

            # Answer/citation/latency metrics
            "faithfulness": faithfulness_score(output, answer, results),
            "answer_relevance": answer_relevance_score(output, answer, row),
            "citation_correctness": citation_correctness_score(output, row, results),
            "citation_hit_rate": citation_hit_rate(output, row, results),
            "latency_ms": output["latency_ms"],

            # Diagnostics only
            "soft_Recall@5": soft_recall_at_k(results, row, TOP_K),
            "max_soft_relevance": max([soft_relevance_score(r, row) for r in results], default=0.0),
            "retrieval_signature": result_signature(results),
            "top_candidate_ids": " || ".join([";".join(r.get("candidate_ids", [])[:3]) for r in results[:3]]),
            "gold_relevant_chunk_ids": row.get("relevant_chunk_ids", []),
            "gold_relevant_document_ids": row.get("relevant_document_ids", []),
            "gold_relevant_pages": row.get("relevant_pages", []),

            "answer_preview": answer[:250],
            "retrieval_type": output.get("retrieval_type"),
            "template_used": output.get("template_used"),
            "graph_hit_count": output.get("graph_hit_count"),
            "blended_chunk_count": output.get("blended_chunk_count"),
            "weight_control_status": output.get("weight_control_status"),
            "weight_attributes_updated": "; ".join(output.get("weight_attributes_updated", [])),
            "using_graphrag_executor": USING_GRAPHRAG_EXECUTOR,
            "graph_db_ready": GRAPH_DB_READY,
            "graph_node_count": GRAPH_NODE_COUNT,
            "run_mode": RUN_MODE,
            "evaluation_status": evaluation_status,
            "eval_source": row.get("eval_source", "d3_gold_qa.csv"),
            "topic_accuracy_basis": topic_accuracy_basis,
            "peft_qlora_mode": "pending_status_cell",
        }

        metric_row["composite_score"] = composite_score(metric_row)

        records.append(metric_row)
        query_method_rows[method_name] = metric_row
        completed.add((idx, method_name))
        save_checkpoints()

    if all(m in query_method_rows for m in ["static_graphrag", "feedback_adaptive_graphrag"]):
        static_score = query_method_rows["static_graphrag"]["composite_score"]
        adaptive_score = query_method_rows["feedback_adaptive_graphrag"]["composite_score"]
        adaptive_helpful = adaptive_score > static_score

        adaptive_row = query_method_rows["feedback_adaptive_graphrag"]

        if adaptive_helpful:
            feedback_reason = "helpful"
        elif adaptive_row["citation_hit_rate"] == 0:
            feedback_reason = "missed_citation"
        elif adaptive_row["document_recall@5"] == 0 and adaptive_row["page_window_recall@5"] == 0:
            feedback_reason = "missed_document_or_page"
        elif adaptive_row["strict_chunk_recall@5"] == 0:
            feedback_reason = "missed_exact_chunk"
        else:
            feedback_reason = "too_broad"

        try:
            update = feedback_adapter.update(helpful=adaptive_helpful, query_topic=routing_topic, reason=feedback_reason)
        except TypeError:
            update = feedback_adapter.update(adaptive_helpful)

        feedback_events.append({
            "index": idx,
            "routing_topic": routing_topic,
            "adaptive_helpful": adaptive_helpful,
            "feedback_reason": feedback_reason,
            "adapter_update": str(update),
        })
        save_checkpoints()

    # Prequential learning: learn only after current query evaluation.
    if topic_accuracy_eligible and gold_true_topic != "unknown":
        try:
            topic_model.learn(query, gold_true_topic)
        except Exception:
            pass

save_checkpoints()

per_query = pd.DataFrame(records)
feedback_df = pd.DataFrame(feedback_events)

print("\nSaved per-query D3 online retrieval results:", PER_QUERY_PATH)
print("Saved feedback events:", FEEDBACK_EVENTS_PATH)
print("Total elapsed minutes:", round((time.perf_counter() - start_all) / 60, 2))
print("Rows:", len(per_query))

display(per_query.head())
display(feedback_df.head())


RUN_MODE=final
Running up to 15 D3 gold queries × 3 methods = 45 GraphRAG calls
Already completed calls: 0

Query 1/15 | gold_topic=technology_innovation | predicted=climate_science (keyword_fallback) | routing=climate_science
Query: FeederBW low-voltage Netze BW one-minute feed-in installations weather metadata 200 feeders
  Running: static_graphrag ... done
  Running: topic_gated_graphrag ... done
  Running: feedback_adaptive_graphrag ... done

Query 2/15 | gold_topic=adaptation | predicted=technology_innovation (river) | routing=technology_innovation
Query: cereal crops phenology sowing date thermal trend photosynthesis starch metabolism phenolo shortens accommodate
  Running: static_graphrag ... Gemini answered with model: gemini-3.1-flash-lite
done
  Running: topic_gated_graphrag ... done
  Running: feedback_adaptive_graphrag ... done

Query 3/15 | gold_topic=adaptation | predicted=technology_innovation (river) | routing=technology_innovation
Query: interactive photogrammetric den

,index,query,reference_answer,true_topic,topic_for_routing,true_topic_source,topic_accuracy_eligible,predicted_topic,prediction_source,topic_correct,...,weight_attributes_updated,using_graphrag_executor,graph_db_ready,graph_node_count,run_mode,evaluation_status,eval_source,topic_accuracy_basis,peft_qlora_mode,composite_score
0,0,FeederBW low-voltage Netze BW one-minute feed-...,The answer should state that FeederBW contains...,technology_innovation,technology_innovation,gold,True,climate_science,keyword_fallback,0,...,GraphRAGExecutor.blender.hybrid_retriever.bm25...,True,True,552,final,final_d3_gold_run,d3_gold_qa.csv,gold_true_topic_only,pending_status_cell,7.5672
1,0,FeederBW low-voltage Netze BW one-minute feed-...,The answer should state that FeederBW contains...,technology_innovation,technology_innovation,gold,True,climate_science,keyword_fallback,0,...,GraphRAGExecutor.blender.hybrid_retriever.bm25...,True,True,552,final,final_d3_gold_run,d3_gold_qa.csv,gold_true_topic_only,pending_status_cell,7.5672
2,0,FeederBW low-voltage Netze BW one-minute feed-...,The answer should state that FeederBW contains...,technology_innovation,technology_innovation,gold,True,climate_science,keyword_fallback,0,...,GraphRAGExecutor.blender.hybrid_retriever.bm25...,True,True,552,final,final_d3_gold_run,d3_gold_qa.csv,gold_true_topic_only,pending_status_cell,7.5672
3,1,cereal crops phenology sowing date thermal tre...,The answer should mention that adjusting sowin...,adaptation,adaptation,gold,True,technology_innovation,river,0,...,GraphRAGExecutor.blender.hybrid_retriever.bm25...,True,True,552,final,final_d3_gold_run,d3_gold_qa.csv,gold_true_topic_only,pending_status_cell,7.5737
4,1,cereal crops phenology sowing date thermal tre...,The answer should mention that adjusting sowin...,adaptation,adaptation,gold,True,technology_innovation,river,0,...,GraphRAGExecutor.blender.hybrid_retriever.bm25...,True,True,552,final,final_d3_gold_run,d3_gold_qa.csv,gold_true_topic_only,pending_status_cell,7.5737


,index,routing_topic,adaptive_helpful,feedback_reason,adapter_update
0,0,climate_science,False,too_broad,"FeedbackUpdate(topic='climate_science', helpfu..."
1,1,technology_innovation,False,too_broad,"FeedbackUpdate(topic='technology_innovation', ..."
2,2,technology_innovation,False,too_broad,"FeedbackUpdate(topic='technology_innovation', ..."
3,3,technology_innovation,False,too_broad,"FeedbackUpdate(topic='technology_innovation', ..."
4,4,technology_innovation,False,too_broad,"FeedbackUpdate(topic='technology_innovation', ..."


## 14. Output audit

This section explains whether zeros are caused by true retrieval failure or ID extraction mismatch.


In [16]:
if "per_query" not in globals():
    if PER_QUERY_PATH.exists():
        per_query = pd.read_csv(PER_QUERY_PATH)
    else:
        raise ValueError("No per_query results found. Run the comparison loop first.")

print("Rows:", len(per_query))
print("Methods:", per_query["method"].value_counts().to_dict())

if "topic_accuracy_eligible" in per_query.columns:
    eligible = per_query["topic_accuracy_eligible"].astype(str).str.lower().isin(["true", "1", "yes"])
    print("\nTopic accuracy basis:", per_query["topic_accuracy_basis"].mode().iloc[0] if "topic_accuracy_basis" in per_query else "unknown")
    print("Rows eligible for topic accuracy:", int(eligible.sum()))
    if eligible.sum() == 0:
        print("Topic accuracy is N/A because true_topic is missing from d3_gold_qa.csv and was inferred only for routing.")

print("\nPredicted topics:")
print(per_query["predicted_topic"].value_counts(dropna=False))

metric_cols = [
    "strict_chunk_recall@5", "strict_chunk_ndcg@5", "strict_chunk_mrr@5",
    "document_recall@5", "document_ndcg@5", "document_mrr@5",
    "page_window_recall@5", "faithfulness", "answer_relevance",
    "citation_correctness", "citation_hit_rate"
]
print("\nZero counts:")
print({c: int((per_query[c] == 0).sum()) for c in metric_cols if c in per_query.columns})

print("\nRetrieval types:")
print(per_query["retrieval_type"].value_counts(dropna=False))

print("\nStrict chunk vs document/page metrics:")
display(
    per_query[[
        "method",
        "strict_chunk_recall@5", "strict_chunk_ndcg@5", "strict_chunk_mrr@5",
        "document_recall@5", "document_ndcg@5", "document_mrr@5",
        "page_window_recall@5",
        "soft_Recall@5", "max_soft_relevance"
    ]].groupby("method").mean(numeric_only=True).round(4)
)

print("\nsoft_Recall@5 caveat:")
print("soft_Recall@5 is diagnostic only. It measures lexical/reference overlap and must not be presented as gold relevance.")

print("\nHow often retrieval ranking changed compared with static:")
sig = per_query.pivot_table(index="index", columns="method", values="retrieval_signature", aggfunc="first")
if {"static_graphrag", "topic_gated_graphrag", "feedback_adaptive_graphrag"}.issubset(sig.columns):
    sig["topic_gated_changed"] = sig["topic_gated_graphrag"] != sig["static_graphrag"]
    sig["adaptive_changed"] = sig["feedback_adaptive_graphrag"] != sig["static_graphrag"]
    display(sig[["static_graphrag", "topic_gated_graphrag", "feedback_adaptive_graphrag", "topic_gated_changed", "adaptive_changed"]])
    print("Topic-gated changed count:", int(sig["topic_gated_changed"].sum()))
    print("Adaptive changed count:", int(sig["adaptive_changed"].sum()))

print("\nSample gold IDs/pages vs returned IDs:")
display(per_query[[
    "index", "method", "strict_chunk_recall@5", "document_recall@5",
    "page_window_recall@5", "retrieval_signature", "top_candidate_ids",
    "gold_relevant_chunk_ids", "gold_relevant_document_ids", "gold_relevant_pages"
]].head(10))


Rows: 45
Methods: {'static_graphrag': 15, 'topic_gated_graphrag': 15, 'feedback_adaptive_graphrag': 15}

Topic accuracy basis: gold_true_topic_only
Rows eligible for topic accuracy: 45

Predicted topics:
predicted_topic
technology_innovation    21
policy_governance        15
mitigation                6
climate_science           3
Name: count, dtype: int64

Zero counts:
{'strict_chunk_recall@5': 9, 'strict_chunk_ndcg@5': 9, 'strict_chunk_mrr@5': 9, 'document_recall@5': 3, 'document_ndcg@5': 3, 'document_mrr@5': 3, 'page_window_recall@5': 6, 'faithfulness': 0, 'answer_relevance': 0, 'citation_correctness': 3, 'citation_hit_rate': 0}

Retrieval types:
retrieval_type
graph_guided       39
hybrid_fallback     6
Name: count, dtype: int64

Strict chunk vs document/page metrics:


,strict_chunk_recall@5,strict_chunk_ndcg@5,strict_chunk_mrr@5,document_recall@5,document_ndcg@5,document_mrr@5,page_window_recall@5,soft_Recall@5,max_soft_relevance
method,,,,,,,,,
feedback_adaptive_graphrag,0.8,0.8,0.8,0.9333,0.8977,0.9,0.8667,1.0,0.8585
static_graphrag,0.8,0.8,0.8,0.9333,0.8977,0.9,0.8667,1.0,0.8585
topic_gated_graphrag,0.8,0.8,0.8,0.9333,0.8977,0.9,0.8667,1.0,0.8585



soft_Recall@5 caveat:
soft_Recall@5 is diagnostic only. It measures lexical/reference overlap and must not be presented as gold relevance.

How often retrieval ranking changed compared with static:


method,static_graphrag,topic_gated_graphrag,feedback_adaptive_graphrag,topic_gated_changed,adaptive_changed
index,,,,,
0,chunk036782 > chunk036783 > chunk036851 > chun...,chunk036782 > chunk036783 > chunk036851 > chun...,chunk036782 > chunk036783 > chunk036851 > chun...,False,False
1,chunk006699 > chunk006694 > chunk006721 > chun...,chunk006699 > chunk006694 > chunk006721 > chun...,chunk006699 > chunk006694 > chunk006721 > chun...,False,False
2,chunk036179 > chunk036162 > chunk036217 > chun...,chunk036179 > chunk036162 > chunk036217 > chun...,chunk036179 > chunk036162 > chunk036217 > chun...,False,False
3,chunk023284 > chunk023283 > chunk036608 > chun...,chunk023284 > chunk023283 > chunk036608 > chun...,chunk023284 > chunk023283 > chunk036608 > chun...,False,False
4,chunk039454 > chunk020520 > chunk039457 > chun...,chunk039454 > chunk020520 > chunk039457 > chun...,chunk039454 > chunk020520 > chunk039457 > chun...,False,False
5,chunk035742 > chunk048106 > chunk035734 > chun...,chunk035742 > chunk048106 > chunk035734 > chun...,chunk035742 > chunk048106 > chunk035734 > chun...,False,False
6,chunk021253 > chunk021229 > chunk021245 > chun...,chunk021253 > chunk021229 > chunk021245 > chun...,chunk021253 > chunk021229 > chunk021245 > chun...,False,False
7,chunk006718 > chunk006699 > chunk006681 > chun...,chunk006718 > chunk006699 > chunk006681 > chun...,chunk006718 > chunk006699 > chunk006681 > chun...,False,False
8,chunk033119 > chunk033151 > chunk033148 > chun...,chunk033119 > chunk033151 > chunk033148 > chun...,chunk033119 > chunk033151 > chunk033148 > chun...,False,False


Topic-gated changed count: 0
Adaptive changed count: 0

Sample gold IDs/pages vs returned IDs:


,index,method,strict_chunk_recall@5,document_recall@5,page_window_recall@5,retrieval_signature,top_candidate_ids,gold_relevant_chunk_ids,gold_relevant_document_ids,gold_relevant_pages
0,0,static_graphrag,1.0,1.0,1.0,chunk036782 > chunk036783 > chunk036851 > chun...,chunk_036782;treutlein_2026_real_world_energy_...,[chunk_036782],[treutlein_2026_real_world_energy_data_200_fee...,[1]
1,0,topic_gated_graphrag,1.0,1.0,1.0,chunk036782 > chunk036783 > chunk036851 > chun...,chunk_036782;treutlein_2026_real_world_energy_...,[chunk_036782],[treutlein_2026_real_world_energy_data_200_fee...,[1]
2,0,feedback_adaptive_graphrag,1.0,1.0,1.0,chunk036782 > chunk036783 > chunk036851 > chun...,chunk_036782;treutlein_2026_real_world_energy_...,[chunk_036782],[treutlein_2026_real_world_energy_data_200_fee...,[1]
3,1,static_graphrag,1.0,1.0,1.0,chunk006699 > chunk006694 > chunk006721 > chun...,chunk_006699;fatima_2020_fingerprints_climate_...,[chunk_006699],[fatima_2020_fingerprints_climate_warming_cere...,[11]
4,1,topic_gated_graphrag,1.0,1.0,1.0,chunk006699 > chunk006694 > chunk006721 > chun...,chunk_006699;fatima_2020_fingerprints_climate_...,[chunk_006699],[fatima_2020_fingerprints_climate_warming_cere...,[11]
5,1,feedback_adaptive_graphrag,1.0,1.0,1.0,chunk006699 > chunk006694 > chunk006721 > chun...,chunk_006699;fatima_2020_fingerprints_climate_...,[chunk_006699],[fatima_2020_fingerprints_climate_warming_cere...,[11]
6,2,static_graphrag,1.0,1.0,1.0,chunk036179 > chunk036162 > chunk036217 > chun...,chunk_036179;ravi_2025_citizen_centered_climat...,[chunk_036179],[ravi_2025_citizen_centered_climate_intelligen...,[7]
7,2,topic_gated_graphrag,1.0,1.0,1.0,chunk036179 > chunk036162 > chunk036217 > chun...,chunk_036179;ravi_2025_citizen_centered_climat...,[chunk_036179],[ravi_2025_citizen_centered_climate_intelligen...,[7]
8,2,feedback_adaptive_graphrag,1.0,1.0,1.0,chunk036179 > chunk036162 > chunk036217 > chun...,chunk_036179;ravi_2025_citizen_centered_climat...,[chunk_036179],[ravi_2025_citizen_centered_climate_intelligen...,[7]
9,3,static_graphrag,1.0,1.0,1.0,chunk023284 > chunk023283 > chunk036608 > chun...,chunk_023284;shah_2025_assessing_climate_vulne...,[chunk_023284],[shah_2025_assessing_climate_vulnerability_ris...,[3]


## 15. PEFT / QLoRA status

This cell does not pretend tuning was integrated. It checks whether tuned output files exist. If not, the final table records `peft_qlora_mode = not_available`.


In [17]:
PEFT_CANDIDATES = [
    Path('/kaggle/working/outputs/d3_or_final_zero_shot_vs_tuned.csv'),
    Path('/kaggle/working/reports/tables/d3_or_final_zero_shot_vs_tuned.csv'),
    Path('/kaggle/input/datasets/aayaehab/d3-or-final-zero-shot-vs-tuned/d3_or_final_zero_shot_vs_tuned.csv'),
    REPORTS_TABLES / "d3_peft_qlora_outputs.csv",
    REPORTS_TABLES / "d3_qlora_outputs.csv",
    REPORTS_TABLES / "d3_zero_shot_vs_tuned.csv",
    REPORTS_TABLES / "d3_or_final_zero_shot_vs_tuned.csv",
    REPORTS_TABLES / "peft_qlora_eval.csv",
]

PEFT_OUTPUT_PATH = None
for candidate in PEFT_CANDIDATES:
    if candidate.exists():
        PEFT_OUTPUT_PATH = candidate
        break

PEFT_QLORA_MODE = "not_available"
peft_comparison = pd.DataFrame()

if PEFT_OUTPUT_PATH is None:
    print("No PEFT/QLoRA tuned output file found.")
    print("PEFT/QLoRA status will be recorded as: not_available")
else:
    print("Found PEFT/QLoRA evidence file:", PEFT_OUTPUT_PATH)
    peft_df = pd.read_csv(PEFT_OUTPUT_PATH)

    display_df = peft_df.copy()
    metric_like_cols = [
        "faithfulness", "answer_relevance", "citation_correct",
        "citation_correctness", "citation_hit_rate", "mean_latency_ms",
        "p95_latency_ms", "latency_ms"
    ]
    for col in metric_like_cols:
        if col in display_df.columns:
            display_df[col] = display_df[col].where(
                display_df[col].notna(),
                "N/A - adapter not trained yet"
            )
    display(display_df.head())

    status_col = find_first_column(peft_df, ["training_status", "status"])
    method_col = find_first_column(peft_df, ["method", "mode", "run_type", "model_type"])

    tuned_rows = pd.DataFrame()
    if method_col:
        tuned_rows = peft_df[peft_df[method_col].astype(str).str.contains("qlora|tuned|adapter", case=False, na=False)]

    tuned_completed = False
    if not tuned_rows.empty and status_col:
        tuned_completed = tuned_rows[status_col].astype(str).str.lower().str.contains("completed|trained|success").any()

    if tuned_completed:
        metric_cols = [c for c in peft_df.columns if c.lower() in {
            "faithfulness", "answer_relevance", "citation_correctness", "citation_correct", "citation_hit_rate", "latency_ms", "mean_latency_ms", "p95_latency_ms"
        }]
        peft_comparison = peft_df.groupby(method_col)[metric_cols].mean(numeric_only=True).reset_index()
        PEFT_QLORA_MODE = "trained_adapter_compared"
        print("PEFT/QLoRA comparison with real tuned-adapter metrics:")
        display(peft_comparison)
    else:
        PEFT_QLORA_MODE = "prepared_not_trained_no_adapter"
        print("PEFT/QLoRA status:")
        print("- Tuning dataset/evidence file exists.")
        print("- QLoRA adapter was NOT trained in the table currently loaded.")
        print("- Tuned-model metrics are shown as N/A, not as zero, because no adapter result exists yet.")
        print("- Do not claim improvement from PEFT/QLoRA unless a GPU training run is completed.")

if "per_query" in globals() and not per_query.empty:
    per_query["peft_qlora_mode"] = PEFT_QLORA_MODE
    per_query.to_csv(PER_QUERY_PATH, index=False)

print("PEFT_QLORA_MODE:", PEFT_QLORA_MODE)


Found PEFT/QLoRA evidence file: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d3_or_final_zero_shot_vs_tuned.csv


,method,training_status,dataset_rows,faithfulness,answer_relevance,citation_correct,mean_latency_ms,p95_latency_ms,model_or_adapter,notes
0,zero_shot_graphrag_baseline,completed_baseline_only,15,0.1912,0.1912,0.2667,10305.12,11824.94,TinyLlama/TinyLlama-1.1B-Chat-v1.0,Base model zero-shot on same 15-row QA set.
1,qlora_tuned_adapter,completed,15,0.2259,0.2259,1.0000,17983.07,18266.50,models/qlora_adapter,Kaggle GPU QLoRA run; metrics are lightweight ...


PEFT/QLoRA comparison with real tuned-adapter metrics:


,method,faithfulness,answer_relevance,citation_correct,mean_latency_ms,p95_latency_ms
0,qlora_tuned_adapter,0.2259,0.2259,1.0000,17983.07,18266.50
1,zero_shot_graphrag_baseline,0.1912,0.1912,0.2667,10305.12,11824.94


PEFT_QLORA_MODE: trained_adapter_compared


## 15. Final comparison table

In [18]:
if "per_query" not in globals() or per_query.empty:
    if PER_QUERY_PATH.exists():
        per_query = pd.read_csv(PER_QUERY_PATH)
    else:
        raise ValueError("No per_query results found. Run the comparison loop first.")

required_methods = {"static_graphrag", "topic_gated_graphrag", "feedback_adaptive_graphrag"}
present_methods = set(per_query["method"].unique())
missing_methods = required_methods - present_methods

print("Methods present:", present_methods)
if missing_methods:
    raise ValueError(f"Missing methods from per_query: {missing_methods}")

# Topic accuracy must not silently count missing true_topic as zero.
if "topic_accuracy_eligible" in per_query.columns:
    eligible = per_query["topic_accuracy_eligible"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    eligible = pd.Series([False] * len(per_query), index=per_query.index)

if eligible.any():
    final_topic_accuracy = pd.to_numeric(per_query.loc[eligible, "topic_correct"], errors="coerce").mean()
    final_topic_accuracy_basis = "gold_true_topic_only"
else:
    final_topic_accuracy = pd.NA
    final_topic_accuracy_basis = "not_available_true_topic_missing"

run_eval_status = per_query["evaluation_status"].mode().iloc[0] if "evaluation_status" in per_query else evaluation_status

if RUN_MODE == "final" and len(per_query["index"].unique()) < 10:
    print("WARNING: RUN_MODE is final, but fewer than 10 unique queries completed.")
    run_eval_status = "partial_d3_run_gemini_or_checkpoint_limited"

summary_rows = []

for method_name, group in per_query.groupby("method", sort=False):
    row = {
        "method": method_name,
        "run_mode": RUN_MODE,
        "evaluation_status": run_eval_status,
        "eval_source": group["eval_source"].iloc[0] if "eval_source" in group else "d3_gold_qa.csv",
        "prequential_topic_accuracy": final_topic_accuracy,
        "topic_accuracy_basis": final_topic_accuracy_basis,

        # Required separated metrics
        "strict_chunk_recall@5": group["strict_chunk_recall@5"].mean(),
        "strict_chunk_ndcg@5": group["strict_chunk_ndcg@5"].mean(),
        "strict_chunk_mrr@5": group["strict_chunk_mrr@5"].mean(),
        "document_recall@5": group["document_recall@5"].mean(),
        "document_ndcg@5": group["document_ndcg@5"].mean(),
        "document_mrr@5": group["document_mrr@5"].mean(),
        "page_window_recall@5": group["page_window_recall@5"].mean(),
        "faithfulness": group["faithfulness"].mean(),
        "answer_relevance": group["answer_relevance"].mean(),
        "citation_correctness": group["citation_correctness"].mean(),
        "citation_hit_rate": group["citation_hit_rate"].mean(),
        "mean_latency_ms": group["latency_ms"].mean(),
        "p95_latency_ms": group["latency_ms"].quantile(0.95),

        # Diagnostics only
        "soft_Recall@5": group["soft_Recall@5"].mean() if "soft_Recall@5" in group else 0.0,
        "max_soft_relevance": group["max_soft_relevance"].mean() if "max_soft_relevance" in group else 0.0,

        "mean_bm25_weight": group["bm25_weight"].mean(),
        "mean_dense_weight": group["dense_weight"].mean(),
        "topic_source": group["topic_source"].iloc[0],
        "adaptation_target": group["adaptation_target"].iloc[0],
        "weight_control_status": (
            group["weight_control_status"].mode().iloc[0]
            if "weight_control_status" in group.columns and not group["weight_control_status"].mode().empty
            else "unknown"
        ),
        "using_graphrag_executor": bool(group["using_graphrag_executor"].iloc[0]),
        "graph_db_ready": bool(group["graph_db_ready"].iloc[0]),
        "graph_node_count": int(group["graph_node_count"].iloc[0]),
        "peft_qlora_mode": PEFT_QLORA_MODE if "PEFT_QLORA_MODE" in globals() else "not_available",
        "n_method_rows": len(group),
    }

    # Backward-compatible aliases for older shared/report code.
    # These aliases are explicitly document-level GraphRAG metrics, not strict chunk metrics.
    row["ndcg@5"] = row["document_ndcg@5"]
    row["mrr@5"] = row["document_mrr@5"]

    summary_rows.append(row)

comparison = pd.DataFrame(summary_rows)

static_scores = per_query[per_query["method"] == "static_graphrag"].set_index("index")["composite_score"]

helps_counts = {}
hurts_counts = {}

for method_name, group in per_query.groupby("method"):
    method_scores = group.set_index("index")["composite_score"]
    common = method_scores.index.intersection(static_scores.index)
    diff = method_scores.loc[common] - static_scores.loc[common]

    helps_counts[method_name] = int((diff > 0).sum())
    hurts_counts[method_name] = int((diff < 0).sum())

comparison["helps_count"] = comparison["method"].map(helps_counts).fillna(0).astype(int)
comparison["hurts_count"] = comparison["method"].map(hurts_counts).fillna(0).astype(int)

ordered_cols = [
    "method",
    "run_mode",
    "evaluation_status",
    "prequential_topic_accuracy",
    "topic_accuracy_basis",
    "strict_chunk_recall@5",
    "strict_chunk_ndcg@5",
    "strict_chunk_mrr@5",
    "document_recall@5",
    "document_ndcg@5",
    "document_mrr@5",
    "page_window_recall@5",
    "ndcg@5",
    "mrr@5",
    "faithfulness",
    "answer_relevance",
    "citation_correctness",
    "citation_hit_rate",
    "mean_latency_ms",
    "p95_latency_ms",
    "helps_count",
    "hurts_count",
    "peft_qlora_mode",
]
extra_cols = [c for c in comparison.columns if c not in ordered_cols]
comparison = comparison[ordered_cols + extra_cols].round(4)

comparison.to_csv(OUTPUT_PATH, index=False)

print("Saved required D3 comparison table:")
print(OUTPUT_PATH)
display(comparison)

print("\nImportant interpretation rules:")
print("- strict_chunk_* is the hardest exact-chunk metric.")
print("- document_* and page_window_recall@5 are fairer GraphRAG evidence metrics.")
print("- soft_Recall@5 is diagnostic only and must not be presented as gold relevance.")
print("- Adaptive improvement should only be claimed if metric deltas or helps_count/hurts_count support it.")


Methods present: {'feedback_adaptive_graphrag', 'topic_gated_graphrag', 'static_graphrag'}
Saved required D3 comparison table:
D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d3_online_retrieval_comparison.csv


,method,run_mode,evaluation_status,prequential_topic_accuracy,topic_accuracy_basis,strict_chunk_recall@5,strict_chunk_ndcg@5,strict_chunk_mrr@5,document_recall@5,document_ndcg@5,...,max_soft_relevance,mean_bm25_weight,mean_dense_weight,topic_source,adaptation_target,weight_control_status,using_graphrag_executor,graph_db_ready,graph_node_count,n_method_rows
0,static_graphrag,final,final_d3_gold_run,0.2667,gold_true_topic_only,0.8,0.8,0.8,0.9333,0.8977,...,0.8585,0.5000,0.5000,none,none,applied,True,True,552,15
1,topic_gated_graphrag,final,final_d3_gold_run,0.2667,gold_true_topic_only,0.8,0.8,0.8,0.9333,0.8977,...,0.8585,0.5267,0.4733,keyword_fallback,BM25/dense weights + graph profile signal,applied,True,True,552,15
2,feedback_adaptive_graphrag,final,final_d3_gold_run,0.2667,gold_true_topic_only,0.8,0.8,0.8,0.9333,0.8977,...,0.8585,0.3933,0.6067,keyword_fallback + FeedbackAdapter,BM25/dense weights; graph profile logged if su...,applied,True,True,552,15



Important interpretation rules:
- strict_chunk_* is the hardest exact-chunk metric.
- document_* and page_window_recall@5 are fairer GraphRAG evidence metrics.
- soft_Recall@5 is diagnostic only and must not be presented as gold relevance.
- Adaptive improvement should only be claimed if metric deltas or helps_count/hurts_count support it.


## 16. Static vs adaptive interpretation

In [19]:
comparison_loaded = pd.read_csv(OUTPUT_PATH)

static = comparison_loaded[comparison_loaded["method"] == "static_graphrag"].iloc[0]
adaptive = comparison_loaded[comparison_loaded["method"] == "feedback_adaptive_graphrag"].iloc[0]
topic_gated = comparison_loaded[comparison_loaded["method"] == "topic_gated_graphrag"].iloc[0]

metric_cols = [
    "strict_chunk_recall@5",
    "strict_chunk_ndcg@5",
    "strict_chunk_mrr@5",
    "document_recall@5",
    "document_ndcg@5",
    "document_mrr@5",
    "page_window_recall@5",
    "faithfulness",
    "answer_relevance",
    "citation_correctness",
    "citation_hit_rate",
    "mean_latency_ms",
    "p95_latency_ms",
]

delta_rows = []
for metric in metric_cols:
    if metric not in comparison_loaded.columns:
        continue
    delta_rows.append({
        "metric": metric,
        "static": static[metric],
        "topic_gated": topic_gated[metric],
        "adaptive": adaptive[metric],
        "topic_gated_minus_static": topic_gated[metric] - static[metric],
        "adaptive_minus_static": adaptive[metric] - static[metric],
    })

delta_df = pd.DataFrame(delta_rows).round(4)
display(delta_df)

adaptive_helps = int(adaptive["helps_count"])
adaptive_hurts = int(adaptive["hurts_count"])

print("Adaptive helps count:", adaptive_helps)
print("Adaptive hurts count:", adaptive_hurts)
print("PEFT/QLoRA status:", adaptive.get("peft_qlora_mode", "unknown"))
print("Evaluation status:", adaptive.get("evaluation_status", "unknown"))
print("Topic accuracy basis:", adaptive.get("topic_accuracy_basis", "unknown"))

quality_metrics = [
    "strict_chunk_recall@5",
    "document_recall@5",
    "document_ndcg@5",
    "document_mrr@5",
    "page_window_recall@5",
    "faithfulness",
    "answer_relevance",
    "citation_correctness",
    "citation_hit_rate",
]
positive_quality_deltas = [
    m for m in quality_metrics
    if m in adaptive.index and m in static.index and adaptive[m] > static[m]
]
negative_quality_deltas = [
    m for m in quality_metrics
    if m in adaptive.index and m in static.index and adaptive[m] < static[m]
]

print("\nHonest adaptive interpretation:")
if adaptive_helps > adaptive_hurts and positive_quality_deltas:
    print("Adaptive shows evidence of improvement over static on this run.")
    print("Positive metric deltas:", positive_quality_deltas)
elif adaptive_helps == adaptive_hurts and not positive_quality_deltas and not negative_quality_deltas:
    print("Adaptive matched static on the measured quality metrics. Do not claim improvement.")
elif adaptive_helps <= adaptive_hurts and not positive_quality_deltas:
    print("Adaptive did not improve over static on this run. Report this honestly.")
else:
    print("Adaptive results are mixed. Report the specific metric deltas and helps/hurts counts rather than claiming a general improvement.")
    print("Positive deltas:", positive_quality_deltas)
    print("Negative deltas:", negative_quality_deltas)

print("\nMetric caveat:")
print("strict_chunk_* metrics are exact gold chunk matching and are intentionally hardest.")
print("document_* and page_window_recall@5 are fairer for GraphRAG because the graph pipeline may retrieve or cite the right document/page without matching the exact chunk ID.")
print("soft_Recall@5 is diagnostic only and must not be presented as gold relevance.")


,metric,static,topic_gated,adaptive,topic_gated_minus_static,adaptive_minus_static
0,strict_chunk_recall@5,0.8000,0.8000,0.8000,0.0000,0.0000
1,strict_chunk_ndcg@5,0.8000,0.8000,0.8000,0.0000,0.0000
2,strict_chunk_mrr@5,0.8000,0.8000,0.8000,0.0000,0.0000
3,document_recall@5,0.9333,0.9333,0.9333,0.0000,0.0000
4,document_ndcg@5,0.8977,0.8977,0.8977,0.0000,0.0000
5,document_mrr@5,0.9000,0.9000,0.9000,0.0000,0.0000
6,page_window_recall@5,0.8667,0.8667,0.8667,0.0000,0.0000
7,faithfulness,0.6794,0.6794,0.6794,0.0000,0.0000
8,answer_relevance,0.7085,0.7085,0.7085,0.0000,0.0000
9,citation_correctness,0.9333,0.9333,0.9333,0.0000,0.0000


Adaptive helps count: 0
Adaptive hurts count: 0
PEFT/QLoRA status: trained_adapter_compared
Evaluation status: final_d3_gold_run
Topic accuracy basis: gold_true_topic_only

Honest adaptive interpretation:
Adaptive matched static on the measured quality metrics. Do not claim improvement.

Metric caveat:
strict_chunk_* metrics are exact gold chunk matching and are intentionally hardest.
document_* and page_window_recall@5 are fairer for GraphRAG because the graph pipeline may retrieve or cite the right document/page without matching the exact chunk ID.
soft_Recall@5 is diagnostic only and must not be presented as gold relevance.


## 17. Shared block kept for later

This notebook does not edit the shared notebook automatically.

Later, copy the final comparison table and interpretation into:

```text
notebooks/D3_graphrag_eval_safety.ipynb
```

under:

```text
Aaya block - Online adaptation inside GraphRAG
```


In [20]:
# -----------------------------
# Update Aaya block in shared D3 notebook
# -----------------------------

SHARED_NOTEBOOK_PATH = ROOT / "notebooks" / "D3_graphrag_eval_safety.ipynb"
MARKER = "Aaya block - Online adaptation inside GraphRAG"

SHARED_BLOCK_DRAFT = """
## Aaya block - Online adaptation inside GraphRAG

Loads d3_online_retrieval_comparison.csv, shows static vs adaptive deltas, helps/hurts counts,
explains strict chunk vs document/page metrics, states that soft metrics are diagnostic only,
and reports PEFT/QLoRA status.
"""

if nbf is None:
    print("nbformat is not installed. Shared notebook update skipped.")
elif not SHARED_NOTEBOOK_PATH.exists():
    print("Shared notebook not found. Skipped automatic update:")
    print(SHARED_NOTEBOOK_PATH)
else:
    shared_cells = [
        nbf.v4.new_markdown_cell(
            "## Aaya block - Online adaptation inside GraphRAG\n\n"
            "This block summarizes Aaya's D3 online/adaptive retrieval comparison inside GraphRAG. "
            "It compares static GraphRAG, topic-gated GraphRAG, and feedback-adaptive GraphRAG using the D3 gold Q/A evaluation table."
        ),
        nbf.v4.new_code_cell(
            "from pathlib import Path\n"
            "import pandas as pd\n\n"
            "try:\n"
            "    from IPython.display import display\n"
            "except Exception:\n"
            "    display = print\n\n"
            "ROOT = Path.cwd().resolve()\n"
            "if ROOT.name.lower() in {'notebooks', 'notebook'}:\n"
            "    ROOT = ROOT.parent\n\n"
            "comparison_path = ROOT / 'reports' / 'tables' / 'd3_online_retrieval_comparison.csv'\n"
            "per_query_path = ROOT / 'reports' / 'tables' / 'd3_online_retrieval_per_query.csv'\n"
            "feedback_path = ROOT / 'reports' / 'tables' / 'd3_online_feedback_events.csv'\n\n"
            "comparison = pd.read_csv(comparison_path)\n"
            "display(comparison)\n"
        ),
        nbf.v4.new_code_cell(
            "static = comparison[comparison['method'] == 'static_graphrag'].iloc[0]\n"
            "adaptive = comparison[comparison['method'] == 'feedback_adaptive_graphrag'].iloc[0]\n"
            "topic_gated = comparison[comparison['method'] == 'topic_gated_graphrag'].iloc[0]\n\n"
            "metrics = [\n"
            "    'strict_chunk_recall@5', 'strict_chunk_ndcg@5', 'strict_chunk_mrr@5',\n"
            "    'document_recall@5', 'document_ndcg@5', 'document_mrr@5',\n"
            "    'page_window_recall@5', 'faithfulness', 'answer_relevance',\n"
            "    'citation_correctness', 'citation_hit_rate', 'mean_latency_ms', 'p95_latency_ms'\n"
            "]\n\n"
            "delta = pd.DataFrame([\n"
            "    {\n"
            "        'metric': metric,\n"
            "        'static_graphrag': static[metric],\n"
            "        'topic_gated_graphrag': topic_gated[metric],\n"
            "        'feedback_adaptive_graphrag': adaptive[metric],\n"
            "        'adaptive_minus_static': adaptive[metric] - static[metric],\n"
            "    }\n"
            "    for metric in metrics if metric in comparison.columns\n"
            "]).round(4)\n\n"
            "display(delta)\n"
        ),
        nbf.v4.new_code_cell(
            "cols = [\n"
            "    'method', 'run_mode', 'evaluation_status', 'helps_count', 'hurts_count',\n"
            "    'prequential_topic_accuracy', 'topic_accuracy_basis',\n"
            "    'strict_chunk_recall@5', 'document_recall@5', 'document_ndcg@5',\n"
            "    'document_mrr@5', 'page_window_recall@5', 'citation_correctness',\n"
            "    'peft_qlora_mode', 'n_method_rows'\n"
            "]\n"
            "cols = [c for c in cols if c in comparison.columns]\n"
            "display(comparison[cols])\n\n"
            "adaptive = comparison[comparison['method'] == 'feedback_adaptive_graphrag'].iloc[0]\n"
            "static = comparison[comparison['method'] == 'static_graphrag'].iloc[0]\n\n"
            "print(f\"Adaptive helped {int(adaptive.get('helps_count', 0))} queries and hurt {int(adaptive.get('hurts_count', 0))} queries compared with static GraphRAG.\")\n"
            "print('PEFT/QLoRA status:', adaptive.get('peft_qlora_mode', 'unknown'))\n"
            "print('Evaluation status:', adaptive.get('evaluation_status', 'unknown'))\n"
            "print('Topic accuracy basis:', adaptive.get('topic_accuracy_basis', 'unknown'))\n\n"
            "if int(adaptive.get('helps_count', 0)) > int(adaptive.get('hurts_count', 0)):\n"
            "    print('Adaptive has more helped than hurt queries, but metric deltas should still be checked before claiming improvement.')\n"
            "else:\n"
            "    print('Adaptive does not show a clear helps-count improvement over static in this run.')\n"
        ),
        nbf.v4.new_markdown_cell(
            "### Aaya interpretation and limitation\n\n"
            "Strict chunk-level retrieval is the hardest metric because it requires the returned evidence to match the exact gold chunk ID. "
            "For GraphRAG, document-level and page-window relevance are also reported because the graph pipeline may retrieve or cite the right document/page without selecting the exact annotated chunk. "
            "Soft overlap metrics, if present in the per-query file, are diagnostic only and should not be presented as gold relevance.\n\n"
            "The feedback-adaptive condition is connected to FeedbackAdapter and updates the BM25/dense retrieval policy. "
            "Adaptive improvement should only be claimed if helps_count/hurts_count and metric deltas support it. "
            "If adaptive does not improve over static, the result should be reported honestly as evidence that topic-level feedback was too coarse or that the GraphRAG retriever did not expose enough controllable behavior. "
            "PEFT/QLoRA status is reported from the comparison table; if tuned outputs are unavailable, tuning remains pending rather than fabricated."
        ),
    ]

    shared_nb = nbf.read(SHARED_NOTEBOOK_PATH, as_version=4)

    start_idx = None
    for i, cell in enumerate(shared_nb.cells):
        if MARKER in "".join(cell.get("source", "")):
            start_idx = i
            break

    if start_idx is None:
        shared_nb.cells.extend(shared_cells)
        action = "appended"
    else:
        end_idx = start_idx + 1
        while end_idx < len(shared_nb.cells):
            src = "".join(shared_nb.cells[end_idx].get("source", ""))
            if src.startswith("# ") or (src.startswith("## ") and MARKER not in src):
                break
            end_idx += 1
        shared_nb.cells[start_idx:end_idx] = shared_cells
        action = "updated"

    nbf.write(shared_nb, SHARED_NOTEBOOK_PATH)
    print(f"Shared notebook {action}:", SHARED_NOTEBOOK_PATH)


Shared notebook updated: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\notebooks\D3_graphrag_eval_safety.ipynb


## Member section export

This writes Aaya's D3 report section to:

```text
reports/member_sections/aaya_d3_adaptation_section.md
```


In [21]:
if not OUTPUT_PATH.exists():
    raise FileNotFoundError("Run the final comparison table cell before exporting the member section.")

comparison_for_section = pd.read_csv(OUTPUT_PATH)

static = comparison_for_section[comparison_for_section["method"] == "static_graphrag"].iloc[0]
adaptive = comparison_for_section[comparison_for_section["method"] == "feedback_adaptive_graphrag"].iloc[0]
topic_gated = comparison_for_section[comparison_for_section["method"] == "topic_gated_graphrag"].iloc[0]

adaptive_helps = int(adaptive.get("helps_count", 0))
adaptive_hurts = int(adaptive.get("hurts_count", 0))

key_metrics = [
    "strict_chunk_recall@5",
    "document_recall@5",
    "document_ndcg@5",
    "document_mrr@5",
    "page_window_recall@5",
    "faithfulness",
    "answer_relevance",
    "citation_correctness",
    "citation_hit_rate",
]

positive_deltas = [m for m in key_metrics if m in adaptive.index and m in static.index and adaptive[m] > static[m]]
negative_deltas = [m for m in key_metrics if m in adaptive.index and m in static.index and adaptive[m] < static[m]]

if adaptive_helps > adaptive_hurts and positive_deltas:
    adaptive_sentence = (
        "The feedback-adaptive run showed evidence of improvement over the static baseline "
        f"on this run, with positive deltas in: {', '.join(positive_deltas)}."
    )
elif adaptive_helps == adaptive_hurts and not positive_deltas and not negative_deltas:
    adaptive_sentence = (
        "The feedback-adaptive run matched the static baseline on the measured quality metrics. "
        "Therefore, no improvement is claimed."
    )
elif adaptive_helps <= adaptive_hurts and not positive_deltas:
    adaptive_sentence = (
        "The feedback-adaptive run did not improve over the static baseline on this run. "
        "This is reported honestly rather than presented as a positive result."
    )
else:
    adaptive_sentence = (
        "The feedback-adaptive results were mixed. The report should discuss the exact metric deltas "
        "and helps/hurts counts instead of claiming a general improvement."
    )

section_text = f"""# Aaya D3 Contribution — Online Adaptation inside GraphRAG

## Scope

My D3 contribution extends the D2 online retrieval work into the GraphRAG pipeline by comparing three retrieval modes:

1. **static_graphrag** — fixed BM25/dense retrieval weights.
2. **topic_gated_graphrag** — topic-based routing profiles from the online topic prediction layer.
3. **feedback_adaptive_graphrag** — River topic prediction plus FeedbackAdapter updates to adjust the retrieval policy over the query stream.

The adaptive component changes the BM25/dense retrieval policy inside the GraphRAG executor. It does not claim to learn new Cypher templates or graph traversal actions.

## Evaluation status

- Run mode: `{adaptive.get("run_mode", RUN_MODE)}`
- Evaluation status: `{adaptive.get("evaluation_status", "unknown")}`
- Topic accuracy basis: `{adaptive.get("topic_accuracy_basis", "unknown")}`
- PEFT/QLoRA status: `{adaptive.get("peft_qlora_mode", "unknown")}`
- Rows per method: `{adaptive.get("n_method_rows", "unknown")}`

If the evaluation status is partial, the results should be treated as limited evidence rather than a complete final gold-set evaluation.

## Main results

| Method | strict_chunk_recall@5 | strict_chunk_ndcg@5 | strict_chunk_mrr@5 | document_recall@5 | document_ndcg@5 | document_mrr@5 | page_window_recall@5 | faithfulness | answer_relevance | citation_correctness | citation_hit_rate |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| static_graphrag | {static['strict_chunk_recall@5']:.4f} | {static['strict_chunk_ndcg@5']:.4f} | {static['strict_chunk_mrr@5']:.4f} | {static['document_recall@5']:.4f} | {static['document_ndcg@5']:.4f} | {static['document_mrr@5']:.4f} | {static['page_window_recall@5']:.4f} | {static['faithfulness']:.4f} | {static['answer_relevance']:.4f} | {static['citation_correctness']:.4f} | {static['citation_hit_rate']:.4f} |
| topic_gated_graphrag | {topic_gated['strict_chunk_recall@5']:.4f} | {topic_gated['strict_chunk_ndcg@5']:.4f} | {topic_gated['strict_chunk_mrr@5']:.4f} | {topic_gated['document_recall@5']:.4f} | {topic_gated['document_ndcg@5']:.4f} | {topic_gated['document_mrr@5']:.4f} | {topic_gated['page_window_recall@5']:.4f} | {topic_gated['faithfulness']:.4f} | {topic_gated['answer_relevance']:.4f} | {topic_gated['citation_correctness']:.4f} | {topic_gated['citation_hit_rate']:.4f} |
| feedback_adaptive_graphrag | {adaptive['strict_chunk_recall@5']:.4f} | {adaptive['strict_chunk_ndcg@5']:.4f} | {adaptive['strict_chunk_mrr@5']:.4f} | {adaptive['document_recall@5']:.4f} | {adaptive['document_ndcg@5']:.4f} | {adaptive['document_mrr@5']:.4f} | {adaptive['page_window_recall@5']:.4f} | {adaptive['faithfulness']:.4f} | {adaptive['answer_relevance']:.4f} | {adaptive['citation_correctness']:.4f} | {adaptive['citation_hit_rate']:.4f} |

## Interpretation

{adaptive_sentence}

The helps/hurts analysis shows that feedback-adaptive GraphRAG helped `{adaptive_helps}` queries and hurt `{adaptive_hurts}` queries compared with the static GraphRAG baseline.

## Metric explanation

Strict chunk-level retrieval is the hardest metric because it only gives credit when GraphRAG retrieves the exact gold chunk ID. Document-level and page-window relevance are also reported because GraphRAG may retrieve evidence from the correct document or page while not matching the exact annotated chunk ID. Soft overlap metrics are used only as diagnostics and are not reported as gold relevance.

## PEFT/QLoRA status

PEFT/QLoRA status is recorded as `{adaptive.get("peft_qlora_mode", "unknown")}`. If tuned output files are unavailable, the notebook does not fabricate a zero-shot vs tuned comparison.

## Limitation

The feedback signal is topic-level and therefore coarse. It can indicate whether an adaptive retrieval policy was useful, but it does not identify whether an error came from BM25 retrieval, dense retrieval, graph expansion, citation selection, or answer generation. Direct graph-query adaptation would require learning graph-specific actions such as relation filters, hop depth, entity constraints, or template selection.
"""

MEMBER_SECTION_PATH.write_text(section_text, encoding="utf-8")
print("Wrote member section:", MEMBER_SECTION_PATH)
print(section_text[:1200])


Wrote member section: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\member_sections\aaya_d3_adaptation_section.md
# Aaya D3 Contribution — Online Adaptation inside GraphRAG

## Scope

My D3 contribution extends the D2 online retrieval work into the GraphRAG pipeline by comparing three retrieval modes:

1. **static_graphrag** — fixed BM25/dense retrieval weights.
2. **topic_gated_graphrag** — topic-based routing profiles from the online topic prediction layer.
3. **feedback_adaptive_graphrag** — River topic prediction plus FeedbackAdapter updates to adjust the retrieval policy over the query stream.

The adaptive component changes the BM25/dense retrieval policy inside the GraphRAG executor. It does not claim to learn new Cypher templates or graph traversal actions.

## Evaluation status

- Run mode: `final`
- Evaluation status: `final_d3_gold_run`
- Topic accuracy basis: `gold_true_topic_only`
- PEFT/QLoRA status: `traine

## Final notes

For final-submission readiness:

1. Run `RUN_MODE = "smoke"` first.
2. If smoke passes, set `RUN_MODE = "final"`.
3. If `data/gold/d3_gold_qa.csv` is missing or incomplete, the notebook will mark the run as partial instead of fabricating final metrics.
4. If Gemini quota stops the run, set `CLEAR_PREVIOUS_RESULTS = False` and rerun later to resume from checkpoints.
5. If strict chunk metrics remain zero, use the document/page metrics and the audit table to explain whether the issue is exact chunk mismatch, missing chunk IDs, or retriever failure.
6. Submit the notebook and generated outputs listed at the top.
